In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===============================
# STEP 1 — Setup CSEE Dataset
# ===============================

from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/SLA_Project_CSEE"

SOURCE_FILE = os.path.join(BASE_DIR, "csee_train.txt")

raw_file = os.path.join(BASE_DIR, "csee_raw_essays.txt")

with open(SOURCE_FILE, "r", encoding="utf-8") as src, open(raw_file, "w", encoding="utf-8") as dst:
    for line in src:
        text = line.strip()
        if text:
            dst.write(text + "\n")

print("Saved raw essays to:", raw_file)

In [ ]:
# ===============================
# STEP 2 — Split Essays into Sentences
# ===============================

import re
import os

BASE_DIR = "/content/drive/MyDrive/SLA_Project_CSEE"
raw_file = os.path.join(BASE_DIR, "csee_raw_essays.txt")
sent_file = os.path.join(BASE_DIR, "csee_sentences.txt")

sentences = []

with open(raw_file, "r", encoding="utf-8") as f:
    for line in f:
        # simple sentence split
        splits = re.split(r'[.!?]+', line.strip())
        for s in splits:
            s = s.strip()
            if len(s.split()) >= 5:   # remove very short fragments
                sentences.append(s)

with open(sent_file, "w", encoding="utf-8") as f:
    for s in sentences:
        f.write(s + "\n")

print("Total sentences:", len(sentences))
print("Saved to:", sent_file)
print("Example:", sentences[0])

In [ ]:
# ===============================
# STEP 3 — Train/Test Split (80/20)
# ===============================

import random
import os

BASE_DIR = "/content/drive/MyDrive/SLA_Project_CSEE"
sent_file = os.path.join(BASE_DIR, "csee_sentences.txt")

with open(sent_file, "r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f if line.strip()]

random.seed(42)
random.shuffle(sentences)

split_idx = int(0.8 * len(sentences))

train_sentences = sentences[:split_idx]
test_sentences  = sentences[split_idx:]

train_file = os.path.join(BASE_DIR, "csee_train.txt")
test_file  = os.path.join(BASE_DIR, "csee_test.txt")

with open(train_file, "w", encoding="utf-8") as f:
    for s in train_sentences:
        f.write(s + "\n")

with open(test_file, "w", encoding="utf-8") as f:
    for s in test_sentences:
        f.write(s + "\n")

print("Train size:", len(train_sentences))
print("Test size :", len(test_sentences))
print("Saved to Drive.")

In [ ]:
# ============================================
# Count total words in dataset
# ============================================

file_path = "/content/drive/MyDrive/SLA_Project_CSEE/csee_sentences.txt"

total_words = 0
total_sentences = 0

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            words = line.split()
            total_words += len(words)
            total_sentences += 1

print("Total sentences:", total_sentences)
print("Total words:", total_words)

In [ ]:
train_path = "/content/drive/MyDrive/SLA_Project_CSEE/csee_train.txt"
test_path  = "/content/drive/MyDrive/SLA_Project_CSEE/csee_test.txt"

def count_words(file_path):
    total_words = 0
    total_sentences = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                words = line.split()
                total_words += len(words)
                total_sentences += 1

    return total_sentences, total_words


train_sent, train_words = count_words(train_path)
test_sent, test_words = count_words(test_path)

print("Train sentences:", train_sent)
print("Train words:", train_words)

print("Test sentences:", test_sent)
print("Test words:", test_words)

Now we fine-tune mBERT on CSEE learner English (MLM objective).


In [ ]:
import os
import math
import torch
from datasets import Dataset

In [ ]:
# =========================================
# STEP 4 — Fine-tune BERT on CSEE (MLM)
# =========================================

import os
import math
import torch
from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

BASE_MODEL = "bert-base-uncased"
BASE_DIR = "/content/drive/MyDrive/SLA_Project_CSEE"

TRAIN_FILE = os.path.join(BASE_DIR, "csee_train.txt")
TEST_FILE  = os.path.join(BASE_DIR, "csee_test.txt")

OUT_DIR = os.path.join(BASE_DIR, "bert_csee_ft")
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Load sentences
# -----------------------------
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_lines = [l.strip() for l in f if l.strip()]

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_lines = [l.strip() for l in f if l.strip()]

print("Train sentences:", len(train_lines))
print("Test sentences :", len(test_lines))

train_dataset = Dataset.from_dict({"text": train_lines})
test_dataset  = Dataset.from_dict({"text": test_lines})

# -----------------------------
# Tokenizer
# -----------------------------
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
test_dataset  = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# -----------------------------
# MLM Collator
# -----------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# -----------------------------
# Load Base Model
# -----------------------------
model = BertForMaskedLM.from_pretrained(BASE_MODEL).to(device)

# -----------------------------
# Training Arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

print("Training BERT on CSEE learner English...")
trainer.train()

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

print("Fine-tuned model saved to:", OUT_DIR)

In [ ]:
from transformers import BertForMaskedLM, BertTokenizer

MODEL_DIR = "/content/drive/MyDrive/SLA_Project_CSEE/bert_csee_ft"

tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model = BertForMaskedLM.from_pretrained(MODEL_DIR)

print("Model loaded successfully")

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# ============================================
# Evaluate BERT Models (Pseudo Perplexity)
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import BertForMaskedLM, BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Load Tokenizer (same as training)
# --------------------------------------------
BASE_MODEL = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------
def compute_pseudo_perplexity(text, tokenizer, model, max_length=64):

    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():

        for i in range(1, seq_len - 1):

            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)

            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)

    return ppl


# --------------------------------------------
# Models to Evaluate
# --------------------------------------------

model_paths = [
    "bert-base-uncased",   # BASE BERT
    "/content/drive/MyDrive/SLA_Project_CSEE/bert_csee_ft"  # Fine-tuned BERT
]


# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------

essay_files = {
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# Loop Over Models
# ============================================

for model_path in model_paths:

    print("\n========================================")
    print("Loading model:", model_path)
    print("========================================")

    model = BertForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        if group == "Chinese_ICNALE":

            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:

            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):

            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)

        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_path] = seed_results


# ============================================
# Final Results
# ============================================

print("\n\nFINAL RESULTS")
print("========================================")

for group in essay_files.keys():

    values = [all_results[m][group] for m in model_paths]

    mean_ppl = np.mean(values)

    print(f"{group}: Mean PPL = {mean_ppl:.3f}")

In [ ]:
!ls /content/drive/MyDrive/SLA_Project_CSEE/bert_csee_ft

In [ ]:
!ls /content/drive/MyDrive | grep -i mandarin

In [ ]:
!find /content/drive/MyDrive -maxdepth 2 -name "*answer*"

In [ ]:
import pandas as pd

file = "/content/drive/MyDrive/Mandarin_BERT_normalized_scores.csv"

df = pd.read_csv(file)

options = df.columns[1:]

answers = []

for i in range(len(df)):
    allowed = []
    for o in options:
        if df.loc[i, o] > 0:
            allowed.append(o)
    answers.append(",".join(allowed))

with open("/content/drive/MyDrive/Mandarin_answers.txt","w") as f:
    for a in answers:
        f.write(a+"\n")

print("Mandarin_answers.txt recreated successfully.")

In [ ]:
# Full ONE-CELL code for CSEE BERT model (same structure)

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK_FIXED.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- YOUR TRAINED MODEL ----
MODEL_DIRS = [
    "/content/drive/MyDrive/SLA_Project_CSEE/bert_csee_ft"
]

OPTIONS = [
    "on","at","like","as","with","inside","of","among","in",
    "by","from","during","about","near","out","round","until",
    "along","for","against","across","to","off","upon","towards",
    "under","around","behind","besides","within","beside","into",
    "between","up","over","before","above","down","after","till",
    "toward","without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES
# -------------------------------------------------

sentences = []

with open(INPUT_FILE,"r",encoding="utf-8") as f:
    for line in f:
        line=line.strip()

        if not line:
            continue

        line=line.replace("____","[MASK]")
        line=line.replace(" ___ "," [MASK] ")
        line=line.replace("___","[MASK]")
        line=line.replace("__","[MASK]")

        if "[MASK]" not in line:
            parts=line.split()

            if len(parts)>1:
                parts.insert(-1,"[MASK]")
                line=" ".join(parts)
            else:
                line=line+" [MASK]"

        sentences.append(line)

print("Loaded",len(sentences),"sentences.")

# -------------------------------------------------
# LOAD OPTION LINES
# -------------------------------------------------

with open(OPTIONS_FILE,"r",encoding="utf-8") as f:
    option_lines=[line.strip() for line in f if line.strip()]

if len(option_lines)!=len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------

for model_dir in MODEL_DIRS:

    model_name=os.path.basename(model_dir.rstrip("/"))

    print("\n========================================")
    print("Running model:",model_name)
    print("Model path:",model_dir)
    print("========================================")

    tokenizer=AutoTokenizer.from_pretrained(model_dir,use_fast=True)
    model=AutoModelForMaskedLM.from_pretrained(model_dir)

    model.to(device)
    model.eval()

    option_token_ids={}

    for w in OPTIONS:
        option_token_ids[w]=tokenizer.convert_tokens_to_ids(w)

    mask_token=tokenizer.mask_token
    mask_token_id=tokenizer.mask_token_id

    if mask_token is None:
        raise ValueError("Tokenizer missing mask token")

    # -------------------------------------------------
    # MASK PREDICTION FUNCTION
    # -------------------------------------------------

    def predict_masked_word(sentence):

        sent=sentence.replace("[MASK]",mask_token)

        inputs=tokenizer(sent,return_tensors="pt")
        inputs={k:v.to(device) for k,v in inputs.items()}

        mask_positions=(inputs["input_ids"]==mask_token_id).nonzero(as_tuple=False)

        if mask_positions.numel()==0:
            return {w:0.0 for w in OPTIONS}

        token_idx=int(mask_positions[0,1].item())

        with torch.no_grad():
            logits=model(**inputs).logits

        mask_logits=logits[0,token_idx,:]
        probs=torch.softmax(mask_logits,dim=-1)

        word_probs={}

        for w in OPTIONS:

            wid=option_token_ids[w]

            if wid is None or wid<0 or wid>=probs.size(-1):
                word_probs[w]=0.0
            else:
                word_probs[w]=float(probs[wid].item())

        return word_probs

    # -------------------------------------------------
    # RAW PREDICTIONS
    # -------------------------------------------------

    rows=[]

    for s in sentences:

        probs=predict_masked_word(s)

        row={"Question":s}
        row.update(probs)

        rows.append(row)

    df_raw=pd.DataFrame(rows)

    raw_path=f"{OUTPUT_DIR}/{model_name}_raw.csv"

    df_raw.to_csv(raw_path,index=False,encoding="utf-8-sig")

    print("Raw predictions saved to:",raw_path)

    # -------------------------------------------------
    # NORMALIZED SCORES
    # -------------------------------------------------

    normalized_rows=[]

    for i,allowed in enumerate(option_lines):

        allowed_list=[w.strip() for w in allowed.split(",") if w.strip()]
        sent=sentences[i]

        total=0.0

        for w in allowed_list:
            if w in df_raw.columns:
                total+=float(df_raw.loc[i,w])

        row={"sentence":sent}

        for w in OPTIONS:

            if (w in allowed_list) and (total>0):
                row[w]=float(df_raw.loc[i,w])/total
            else:
                row[w]=0.0

        normalized_rows.append(row)

    df_norm=pd.DataFrame(normalized_rows)

    norm_path=f"{OUTPUT_DIR}/{model_name}_normalized.csv"

    df_norm.to_csv(norm_path,index=False,encoding="utf-8-sig")

    print("Normalized scores saved to:",norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

 Install Modern Grammar Correction Model

In [ ]:
!pip install transformers sentencepiece

Load Grammar Correction Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

MODEL_NAME = "prithivida/grammar_error_correcter_v1"

tokenizer_gec = AutoTokenizer.from_pretrained(MODEL_NAME)
model_gec = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

model_gec.eval()

print("Grammar correction model loaded.")

eep only sentences that changed (i.e., incorrect ones)

In [ ]:
RAW_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/csee_raw_essays.txt"

with open(RAW_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

print("Actual lines in file:", len(lines))

In [ ]:
# Reload file fresh
with open(RAW_FILE, "r", encoding="utf-8") as f:
    learner_sentences = [line.strip() for line in f if line.strip()]

print("Actual lines loaded:", len(learner_sentences))

In [ ]:
import torch
import tqdm
import os

RAW_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/csee_raw_essays.txt"
OUTPUT_INCORRECT = "/content/drive/MyDrive/SLA_Project_CSEE/csee_incorrect_only.txt"

BATCH_SIZE = 32  # you can increase to 48 if GPU memory allows

# ----------------------------
# Load learner sentences
# ----------------------------
with open(RAW_FILE, "r", encoding="utf-8") as f:
    learner_sentences = [line.strip() for line in f if line.strip()]

print("Total learner sentences:", len(learner_sentences))

incorrect_sentences = []

print("Running FAST grammar correction on FULL dataset...")

# ----------------------------
# Batch Processing
# ----------------------------
for i in tqdm.tqdm(range(0, len(learner_sentences), BATCH_SIZE)):

    batch = learner_sentences[i:i+BATCH_SIZE]

    inputs = tokenizer_gec(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model_gec.generate(
            **inputs,
            max_length=128,
            num_beams=2,   # reduced from 4 → much faster
        )

    decoded = tokenizer_gec.batch_decode(outputs, skip_special_tokens=True)

    # Compare original vs corrected
    for original, corrected in zip(batch, decoded):
        if corrected.strip().lower() != original.strip().lower():
            incorrect_sentences.append(original)

print("Total incorrect sentences:", len(incorrect_sentences))

# ----------------------------
# Save to Drive
# ----------------------------
os.makedirs(os.path.dirname(OUTPUT_INCORRECT), exist_ok=True)

with open(OUTPUT_INCORRECT, "w", encoding="utf-8") as f:
    for s in incorrect_sentences:
        f.write(s + "\n")

print("Saved incorrect sentences to:", OUTPUT_INCORRECT)

Create Incorrect 80/20 Split

In [ ]:
import random
import os

INCORRECT_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/csee_incorrect_only.txt"
SAVE_DIR = "/content/drive/MyDrive/SLA_Project_CSEE/splits"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(INCORRECT_FILE, "r", encoding="utf-8") as f:
    incorrect_sentences = [line.strip() for line in f if line.strip()]

print("Total incorrect sentences:", len(incorrect_sentences))

random.seed(42)
random.shuffle(incorrect_sentences)

split_idx = int(0.8 * len(incorrect_sentences))

incorrect_train = incorrect_sentences[:split_idx]
incorrect_test = incorrect_sentences[split_idx:]

print("Incorrect Train:", len(incorrect_train))
print("Incorrect Test:", len(incorrect_test))

with open(f"{SAVE_DIR}/incorrect_train.txt", "w", encoding="utf-8") as f:
    for s in incorrect_train:
        f.write(s + "\n")

with open(f"{SAVE_DIR}/incorrect_test.txt", "w", encoding="utf-8") as f:
    for s in incorrect_test:
        f.write(s + "\n")

print("Incorrect splits saved.")

Train BERT on Incorrect-Only Train Split

In [ ]:
import os
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

In [ ]:
import torch
import math
from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SAVE_DIR = "/content/drive/MyDrive/SLA_Project_CSEE/models"
TRAIN_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/splits/incorrect_train.txt"
MODEL_SAVE_PATH = "/content/drive/MyDrive/SLA_Project_CSEE/models/csee_incorrect_bert"

# Load training data
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_lines = [line.strip() for line in f if line.strip()]

print("Training sentences:", len(train_lines))

dataset = Dataset.from_dict({"text": train_lines})

# -------- CHANGED MODEL HERE --------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# -------- CHANGED MODEL HERE --------
model = BertForMaskedLM.from_pretrained("bert-base-uncased")
model.to(DEVICE)

training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    save_strategy="epoch",
    logging_steps=500,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Training BERT on INCORRECT sentences only...")
trainer.train()

trainer.save_model(MODEL_SAVE_PATH)

print("Model saved to:", MODEL_SAVE_PATH)

In [ ]:
!ls /content/drive/MyDrive/SLA_Project_CSEE/models

Perplexity on Incorrect Test Set

In [ ]:
# ============================================
# Evaluate BERT Models (Pseudo Perplexity)
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import BertForMaskedLM, BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Load Tokenizer (same as training)
# --------------------------------------------

BASE_MODEL = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------

def compute_pseudo_perplexity(text, tokenizer, model, max_length=64):

    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():

        for i in range(1, seq_len - 1):

            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)

            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)

    return ppl


# --------------------------------------------
# Models to Evaluate
# --------------------------------------------

model_paths = [

    "bert-base-uncased",  # Base BERT

    "/content/drive/MyDrive/SLA_Project_CSEE/models/csee_incorrect_bert"  # Incorrect-only BERT
]


# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------

essay_files = {
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# Loop Over Models
# ============================================

for model_path in model_paths:

    print("\n========================================")
    print("Loading model:", model_path)
    print("========================================")

    model = BertForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        if group == "Chinese_ICNALE":

            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:

            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):

            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)

        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_path] = seed_results


# ============================================
# Final Results
# ============================================

print("\n\nFINAL RESULTS")
print("========================================")

for group in essay_files.keys():

    values = [all_results[m][group] for m in model_paths]

    mean_ppl = np.mean(values)

    print(f"{group}: Mean PPL = {mean_ppl:.3f}")

In [ ]:
# Full ONE-CELL code for Incorrect BERT model (same structure)

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK_FIXED.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- INCORRECT BERT MODEL ----
MODEL_DIRS = [
    "/content/drive/MyDrive/SLA_Project_CSEE/models/csee_incorrect_bert"
]

OPTIONS = [
    "on","at","like","as","with","inside","of","among","in",
    "by","from","during","about","near","out","round","until",
    "along","for","against","across","to","off","upon","towards",
    "under","around","behind","besides","within","beside","into",
    "between","up","over","before","above","down","after","till",
    "toward","without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES
# -------------------------------------------------

sentences = []

with open(INPUT_FILE,"r",encoding="utf-8") as f:
    for line in f:
        line=line.strip()

        if not line:
            continue

        line=line.replace("____","[MASK]")
        line=line.replace(" ___ "," [MASK] ")
        line=line.replace("___","[MASK]")
        line=line.replace("__","[MASK]")

        if "[MASK]" not in line:
            parts=line.split()

            if len(parts)>1:
                parts.insert(-1,"[MASK]")
                line=" ".join(parts)
            else:
                line=line+" [MASK]"

        sentences.append(line)

print("Loaded",len(sentences),"sentences.")

# -------------------------------------------------
# LOAD OPTION LINES
# -------------------------------------------------

with open(OPTIONS_FILE,"r",encoding="utf-8") as f:
    option_lines=[line.strip() for line in f if line.strip()]

if len(option_lines)!=len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------

for model_dir in MODEL_DIRS:

    model_name=os.path.basename(model_dir.rstrip("/"))

    print("\n========================================")
    print("Running model:",model_name)
    print("Model path:",model_dir)
    print("========================================")

    tokenizer=AutoTokenizer.from_pretrained(model_dir,use_fast=True)
    model=AutoModelForMaskedLM.from_pretrained(model_dir)

    model.to(device)
    model.eval()

    option_token_ids={}

    for w in OPTIONS:
        option_token_ids[w]=tokenizer.convert_tokens_to_ids(w)

    mask_token=tokenizer.mask_token
    mask_token_id=tokenizer.mask_token_id

    if mask_token is None:
        raise ValueError("Tokenizer missing mask token")

    # -------------------------------------------------
    # MASK PREDICTION FUNCTION
    # -------------------------------------------------

    def predict_masked_word(sentence):

        sent=sentence.replace("[MASK]",mask_token)

        inputs=tokenizer(sent,return_tensors="pt")
        inputs={k:v.to(device) for k,v in inputs.items()}

        mask_positions=(inputs["input_ids"]==mask_token_id).nonzero(as_tuple=False)

        if mask_positions.numel()==0:
            return {w:0.0 for w in OPTIONS}

        token_idx=int(mask_positions[0,1].item())

        with torch.no_grad():
            logits=model(**inputs).logits

        mask_logits=logits[0,token_idx,:]
        probs=torch.softmax(mask_logits,dim=-1)

        word_probs={}

        for w in OPTIONS:

            wid=option_token_ids[w]

            if wid is None or wid<0 or wid>=probs.size(-1):
                word_probs[w]=0.0
            else:
                word_probs[w]=float(probs[wid].item())

        return word_probs

    # -------------------------------------------------
    # RAW PREDICTIONS
    # -------------------------------------------------

    rows=[]

    for s in sentences:

        probs=predict_masked_word(s)

        row={"Question":s}
        row.update(probs)

        rows.append(row)

    df_raw=pd.DataFrame(rows)

    raw_path=f"{OUTPUT_DIR}/{model_name}_raw.csv"

    df_raw.to_csv(raw_path,index=False,encoding="utf-8-sig")

    print("Raw predictions saved to:",raw_path)

    # -------------------------------------------------
    # NORMALIZED SCORES
    # -------------------------------------------------

    normalized_rows=[]

    for i,allowed in enumerate(option_lines):

        allowed_list=[w.strip() for w in allowed.split(",") if w.strip()]
        sent=sentences[i]

        total=0.0

        for w in allowed_list:
            if w in df_raw.columns:
                total+=float(df_raw.loc[i,w])

        row={"sentence":sent}

        for w in OPTIONS:

            if (w in allowed_list) and (total>0):
                row[w]=float(df_raw.loc[i,w])/total
            else:
                row[w]=0.0

        normalized_rows.append(row)

    df_norm=pd.DataFrame(normalized_rows)

    norm_path=f"{OUTPUT_DIR}/{model_name}_normalized.csv"

    df_norm.to_csv(norm_path,index=False,encoding="utf-8-sig")

    print("Normalized scores saved to:",norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

Incorrect-Trained Model → GOLD MASK Test

Load incorrect-only train split
• Load base BERT
• Add LoRA adapters
• Train only LoRA parameters
• Save adapter safely to Drive


In [ ]:
# =========================
# CELL 1 — Setup + paths (LoRA on BASE BERT)
# =========================

from google.colab import drive
drive.mount("/content/drive")

import os, random, math
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# ---- Base model (CHANGED TO ENGLISH BERT) ----
BASE_MODEL = "bert-base-uncased"

# ---- Dataset (Chinese learner English)

TRAIN_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/splits/incorrect_train.txt"
TEST_FILE  = "/content/drive/MyDrive/SLA_Project_CSEE/splits/incorrect_test.txt"

MIXED_TEST_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/csee_test.txt"

# ---- Save LoRA adapter ----
SAVE_DIR = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora"

# (renamed for clarity)
ADAPTER_OUT = f"{SAVE_DIR}/lora_incorrect_base_bert"

os.makedirs(ADAPTER_OUT, exist_ok=True)

print("TRAIN_FILE exists:", os.path.exists(TRAIN_FILE))
print("TEST_FILE exists :", os.path.exists(TEST_FILE))
print("MIXED_TEST_FILE exists:", os.path.exists(MIXED_TEST_FILE))
print("Adapter save path:", ADAPTER_OUT)

Install PEFT (LoRA library)

In [ ]:
# =========================
# CELL 2 — Install PEFT (LoRA)
# =========================

!pip install -q peft accelerate

print("PEFT + Accelerate installed.")

Load Incorrect-Only Dataset

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# Load training sentences
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_lines = [l.strip() for l in f if l.strip()]

print("Training sentences:", len(train_lines))
print("Sample:", train_lines[0])

# Create dataset
train_dataset = Dataset.from_dict({"text": train_lines})

# Tokenizer (use AutoTokenizer)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64,
        return_special_tokens_mask=True   # ⭐ IMPORTANT for MLM
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenization complete.")

 Inject LoRA into Base unacsed BERT

In [ ]:
from transformers import BertForMaskedLM
from peft import LoraConfig, get_peft_model
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load base BERT
base_model = BertForMaskedLM.from_pretrained(BASE_MODEL)

# LoRA config (CORRECT)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"   # ✅ KEEP THIS
)

# Attach LoRA
model = get_peft_model(base_model, lora_config)
model.to(DEVICE)

model.print_trainable_parameters()

Train LoRA on INCORRECT dataset

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Data collator (MANDATORY)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir=ADAPTER_OUT,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-4,
    logging_steps=100,
    save_strategy="epoch",
    report_to="none",
    fp16=True   # 🚀 SPEED BOOST
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print("Training LoRA on incorrect-only Chinese learner English...")
trainer.train()

# Save only LoRA adapter
model.save_pretrained(ADAPTER_OUT)

print("LoRA adapter saved to:", ADAPTER_OUT)

In [ ]:
# Save LoRA adapter (only adapter weights, not full model)
model.save_pretrained(ADAPTER_OUT)

# Save tokenizer too (important for loading later)
tokenizer.save_pretrained(ADAPTER_OUT)

print("LoRA adapter saved to:", ADAPTER_OUT)

# Verify files
import os
print("Saved files:", os.listdir(ADAPTER_OUT))

Perplexity Evaluation (LoRA vs BASE)

In [ ]:
# ============================================
# Evaluate BERT Models (Pseudo Perplexity)
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import BertForMaskedLM, BertTokenizer
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Tokenizer
# --------------------------------------------

BASE_MODEL = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------

def compute_pseudo_perplexity(text, tokenizer, model, max_length=64):

    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():

        for i in range(1, seq_len - 1):

            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)

            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)
    return ppl


# --------------------------------------------
# Models (ONLY BASE + LoRA)
# --------------------------------------------

models_config = {

    "Base_BERT": "bert-base-uncased",

    "LoRA_Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora/lora_incorrect_base_bert"
}


# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------

essay_files = {
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# Loop Over Models
# ============================================

for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading model:", model_name)
    print("========================================")

    # 🔹 Base BERT
    if model_name == "Base_BERT":
        model = BertForMaskedLM.from_pretrained(model_path).to(device)

    # 🔹 LoRA BERT (IMPORTANT)
    elif model_name == "LoRA_Incorrect_BERT":
        base_model = BertForMaskedLM.from_pretrained(BASE_MODEL)
        model = PeftModel.from_pretrained(base_model, model_path)
        model = model.to(device)

    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        if group == "Chinese_ICNALE":

            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:

            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):

            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)

        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_name] = seed_results


# ============================================
# Final Results
# ============================================

print("\n\nFINAL RESULTS")
print("========================================")

for group in essay_files.keys():

    print(f"\n{group}")

    for model_name in models_config.keys():
        print(f"{model_name}: {all_results[model_name][group]:.3f}")

In [ ]:
# Full ONE-CELL code for LoRA Incorrect BERT model

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, BertForMaskedLM
from peft import PeftModel

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK_FIXED.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- LoRA MODEL PATH ----
MODEL_DIRS = [
    "/content/drive/MyDrive/SLA_Project_CSEE/models_lora/lora_incorrect_base_bert"
]

BASE_MODEL = "bert-base-uncased"

OPTIONS = [
    "on","at","like","as","with","inside","of","among","in",
    "by","from","during","about","near","out","round","until",
    "along","for","against","across","to","off","upon","towards",
    "under","around","behind","besides","within","beside","into",
    "between","up","over","before","above","down","after","till",
    "toward","without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES
# -------------------------------------------------

sentences = []

with open(INPUT_FILE,"r",encoding="utf-8") as f:
    for line in f:
        line=line.strip()

        if not line:
            continue

        line=line.replace("____","[MASK]")
        line=line.replace(" ___ "," [MASK] ")
        line=line.replace("___","[MASK]")
        line=line.replace("__","[MASK]")

        if "[MASK]" not in line:
            parts=line.split()

            if len(parts)>1:
                parts.insert(-1,"[MASK]")
                line=" ".join(parts)
            else:
                line=line+" [MASK]"

        sentences.append(line)

print("Loaded",len(sentences),"sentences.")

# -------------------------------------------------
# LOAD OPTION LINES
# -------------------------------------------------

with open(OPTIONS_FILE,"r",encoding="utf-8") as f:
    option_lines=[line.strip() for line in f if line.strip()]

if len(option_lines)!=len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------

for model_dir in MODEL_DIRS:

    model_name = "lora_incorrect_bert"

    print("\n========================================")
    print("Running model:", model_name)
    print("Model path:", model_dir)
    print("========================================")

    # 🔹 Load tokenizer (from base model or adapter — both ok)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

    # 🔹 Load base model
    base_model = BertForMaskedLM.from_pretrained(BASE_MODEL)

    # 🔹 Attach LoRA adapter
    model = PeftModel.from_pretrained(base_model, model_dir)

    model.to(device)
    model.eval()

    option_token_ids = {}

    for w in OPTIONS:
        option_token_ids[w] = tokenizer.convert_tokens_to_ids(w)

    mask_token = tokenizer.mask_token
    mask_token_id = tokenizer.mask_token_id

    if mask_token is None:
        raise ValueError("Tokenizer missing mask token")

    # -------------------------------------------------
    # MASK PREDICTION FUNCTION
    # -------------------------------------------------

    def predict_masked_word(sentence):

        sent = sentence.replace("[MASK]", mask_token)

        inputs = tokenizer(sent, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)

        if mask_positions.numel() == 0:
            return {w: 0.0 for w in OPTIONS}

        token_idx = int(mask_positions[0,1].item())

        with torch.no_grad():
            logits = model(**inputs).logits

        mask_logits = logits[0, token_idx, :]
        probs = torch.softmax(mask_logits, dim=-1)

        word_probs = {}

        for w in OPTIONS:
            wid = option_token_ids[w]

            if wid is None or wid < 0 or wid >= probs.size(-1):
                word_probs[w] = 0.0
            else:
                word_probs[w] = float(probs[wid].item())

        return word_probs

    # -------------------------------------------------
    # RAW PREDICTIONS
    # -------------------------------------------------

    rows = []

    for s in sentences:
        probs = predict_masked_word(s)

        row = {"Question": s}
        row.update(probs)

        rows.append(row)

    df_raw = pd.DataFrame(rows)

    raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
    df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")

    print("Raw predictions saved to:", raw_path)

    # -------------------------------------------------
    # NORMALIZED SCORES
    # -------------------------------------------------

    normalized_rows = []

    for i, allowed in enumerate(option_lines):

        allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]
        sent = sentences[i]

        total = 0.0

        for w in allowed_list:
            if w in df_raw.columns:
                total += float(df_raw.loc[i, w])

        row = {"sentence": sent}

        for w in OPTIONS:
            if (w in allowed_list) and (total > 0):
                row[w] = float(df_raw.loc[i, w]) / total
            else:
                row[w] = 0.0

        normalized_rows.append(row)

    df_norm = pd.DataFrame(normalized_rows)

    norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
    df_norm.to_csv(norm_path, index=False, encoding="utf-8-sig")

    print("Normalized scores saved to:", norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

BUILD TEST LOADER (Incorrect Test Split)

traib BERT on the clean dataset which only have incorrect essay only on very few alyers using LoRA and then in between adding the muse vectors as auxilory in BERT

In [ ]:
# =========================
# CELL 1 — Setup + paths (LoRA + MUSE auxiliary loss)
# =========================

from google.colab import drive
drive.mount("/content/drive")

import os, re, math, random
import torch
import numpy as np

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---- Base model (CHANGED TO ENGLISH BERT) ----
BASE_MODEL = "bert-base-uncased"

# ---- Chinese learner English (incorrect-only) ----
TRAIN_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/splits/incorrect_train.txt"
TEST_FILE  = "/content/drive/MyDrive/SLA_Project_CSEE/splits/incorrect_test.txt"

# (Optional) mixed test (generalization check)
MIXED_TEST_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/csee_test.txt"

# ---- MUSE / FastText resources ----
EN_VEC = "/content/drive/MyDrive/muse_data/wiki.en.vec"
ZH_VEC = "/content/drive/MyDrive/muse_data/cc.zh.300.vec"

# ---- Work directory ----
MUSE_WORKDIR = "/content/drive/MyDrive/muse_data_csee"
os.makedirs(MUSE_WORKDIR, exist_ok=True)

EN_100K = f"{MUSE_WORKDIR}/wiki.en.100k.vec"
ZH_100K = f"{MUSE_WORKDIR}/cc.zh.100k.vec"
DICO_PATH = f"{MUSE_WORKDIR}/en-zh.txt"

# ---- Save outputs ----
OUT_DIR = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse"
os.makedirs(OUT_DIR, exist_ok=True)

ADAPTER_OUT = f"{OUT_DIR}/lora_incorrect_base_bert_muse"
PROJ_OUT    = f"{OUT_DIR}/proj_300_to_768.pt"
MAPPING_OUT = f"{OUT_DIR}/muse_en_zh_mapping.npy"

# ---- Debug paths ----
print("\n--- Paths check ---")
print("TRAIN exists:", os.path.exists(TRAIN_FILE))
print("TEST exists :", os.path.exists(TEST_FILE))
print("MIXED exists:", os.path.exists(MIXED_TEST_FILE))
print("EN_VEC exists:", os.path.exists(EN_VEC))
print("ZH_VEC exists:", os.path.exists(ZH_VEC))
print("Workdir:", MUSE_WORKDIR)
print("Adapter out:", ADAPTER_OUT)
print("Proj out:", PROJ_OUT)
print("Mapping out:", MAPPING_OUT)

Model = Base mBERT + LoRA + MUSE auxiliary loss
Total Loss = MLM_loss + λ * Alignment_loss
	MLM_loss → makes model behave like Chinese learner
	•	Alignment_loss → gently pulls English embeddings toward MUSE-aligned space
	•	λ → small weight (like 0.05 or 0.1)

Build EN→ZH mapping (Procrustes) and save it

In [ ]:
# =========================
# CELL 2 — Build MUSE-style mapping (EN -> ZH) with Procrustes
# =========================

import os
import re
import numpy as np
from pathlib import Path

WORKDIR = "/content/drive/MyDrive/muse_data_csee"
os.makedirs(WORKDIR, exist_ok=True)

EN_VEC_PATH = "/content/drive/MyDrive/muse_data/wiki.en.vec"
ZH_VEC_PATH = "/content/drive/MyDrive/muse_data/cc.zh.300.vec"

DICO_PATH = f"{WORKDIR}/en-zh.txt"

# save under your uncased-BERT MUSE folder
MAPPING_DIR = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse"
os.makedirs(MAPPING_DIR, exist_ok=True)
MAPPING_OUT = f"{MAPPING_DIR}/muse_en_zh_mapping.npy"

# 1) download dictionary if missing
if not os.path.exists(DICO_PATH):
    !wget -O "{DICO_PATH}" "https://dl.fbaipublicfiles.com/arrival/dictionaries/en-zh.txt"

print("Dictionary:", DICO_PATH, "exists:", os.path.exists(DICO_PATH))

def read_fasttext_header(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        header = f.readline().strip().split()
        if len(header) == 2 and header[0].isdigit():
            return int(header[0]), int(header[1]), True
        return None, None, False

def load_fasttext_subset(vec_path, wanted_words, max_read=None):
    """
    Stream-load only vectors for wanted_words from a .vec file.
    """
    wanted = set(wanted_words)
    found = {}
    total_words, dim, has_header = read_fasttext_header(vec_path)

    with open(vec_path, "r", encoding="utf-8", errors="ignore") as f:
        if has_header:
            _ = f.readline()

        for i, line in enumerate(f):
            if max_read and i >= max_read:
                break

            parts = line.rstrip().split(" ")
            if len(parts) < 10:
                continue

            w = parts[0]
            if w in wanted:
                try:
                    vec = np.asarray(parts[1:], dtype=np.float32)
                    found[w] = vec
                except:
                    pass

            if len(found) == len(wanted):
                break

    return found

# 2) read dictionary pairs (limit to keep it fast/stable)
pairs = []
with open(DICO_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        sp = line.split()
        if len(sp) < 2:
            continue

        en, zh = sp[0], sp[1]
        pairs.append((en, zh))

print("Raw dictionary pairs:", len(pairs))

# take a reasonable number (enough for mapping)
MAX_PAIRS = 20000
pairs = pairs[:MAX_PAIRS]

en_words = [p[0] for p in pairs]
zh_words = [p[1] for p in pairs]

# 3) load only vectors for those words
print("Loading EN subset vectors...")
en_vecs = load_fasttext_subset(EN_VEC_PATH, en_words)

print("Loading ZH subset vectors...")
zh_vecs = load_fasttext_subset(ZH_VEC_PATH, zh_words)

# 4) build aligned matrices
X, Y, kept = [], [], 0
for en, zh in pairs:
    if en in en_vecs and zh in zh_vecs:
        X.append(en_vecs[en])
        Y.append(zh_vecs[zh])
        kept += 1

if len(X) == 0:
    raise ValueError("No aligned EN-ZH vector pairs were found. Check vector files and dictionary.")

X = np.stack(X, axis=0)
Y = np.stack(Y, axis=0)

print("Pairs kept (both vectors exist):", kept)
print("X shape:", X.shape, "Y shape:", Y.shape)

# normalize rows
X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
Y = Y / (np.linalg.norm(Y, axis=1, keepdims=True) + 1e-8)

# 5) Orthogonal Procrustes: W = argmin ||XW - Y||
M = X.T @ Y
U, _, Vt = np.linalg.svd(M)
W = U @ Vt

print("Mapping W shape:", W.shape)

np.save(MAPPING_OUT, W)
print("Saved mapping to:", MAPPING_OUT)

Build token-id → mapped FastText targets (and save)

We will create a table for English tokens that exist in:
	•	mBERT tokenizer vocab (single token)
	•	FastText English vocab (we stream-load only needed)
	•	then we map EN vec → ZH space using W


In [ ]:
# =========================
# CELL 3 — Build token_id -> mapped_fasttext_vector(300) target table
# =========================

import numpy as np
import torch
import re
from transformers import BertTokenizer

# ✅ FIXED: use uncased BERT (consistent with your pipeline)
BASE_MODEL = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

MAPPING_OUT = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/muse_en_zh_mapping.npy"
W = np.load(MAPPING_OUT).astype(np.float32)

EN_VEC_PATH = "/content/drive/MyDrive/muse_data/wiki.en.vec"

# Choose which tokenizer tokens to align
def is_good_token(tok: str):
    if tok.startswith("##"):
        return False
    if len(tok) < 2:
        return False
    # ✅ FIXED: lowercase regex for uncased BERT
    if re.fullmatch(r"[a-z]+", tok) is None:
        return False
    return True

vocab = tokenizer.get_vocab()
good_tokens = [t for t in vocab.keys() if is_good_token(t)]
print("Candidate tokenizer tokens:", len(good_tokens))

# ✅ Slightly reduced for speed (optional but recommended)
MAX_TOKENS = 30000
good_tokens = sorted(good_tokens)[:MAX_TOKENS]

print("Using tokens for alignment:", len(good_tokens))

# Uses your previously defined function
en_ft = load_fasttext_subset(EN_VEC_PATH, good_tokens)
print("FastText vectors found:", len(en_ft))

# Build token_id -> mapped_vec (300)
tokenid_to_mapped300 = {}

for tok, vec in en_ft.items():
    tid = vocab.get(tok, None)
    if tid is None:
        continue

    vec = vec.astype(np.float32)
    vec = vec / (np.linalg.norm(vec) + 1e-8)

    mapped = vec @ W
    tokenid_to_mapped300[tid] = mapped

print("token_id targets built:", len(tokenid_to_mapped300))

TARGETS_OUT = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/tokenid_to_mapped300.npz"

# ✅ safety check
if len(tokenid_to_mapped300) == 0:
    raise ValueError("No mappings created — check inputs")

np.savez_compressed(
    TARGETS_OUT,
    token_ids=np.array(list(tokenid_to_mapped300.keys()), dtype=np.int32),
    mapped=np.stack(list(tokenid_to_mapped300.values()), axis=0).astype(np.float32)
)

print("Saved targets:", TARGETS_OUT)

Train projection (300→768) to match BERT embedding space

We train a small linear layer so mapped FastText vectors can live in BERT embedding space.

In [ ]:
# =========================
# CELL 4 — Train projection 300 -> 768
# =========================

import torch
import torch.nn as nn
import numpy as np
from transformers import BertForMaskedLM, BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ FIXED: use uncased BERT (consistent everywhere)
BASE_MODEL = "bert-base-uncased"

model_base = BertForMaskedLM.from_pretrained(BASE_MODEL).to(device)
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

TARGETS_OUT = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/tokenid_to_mapped300.npz"
PROJ_OUT = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/proj_300_to_768.pt"

# -------------------------------------------------
# Load data
# -------------------------------------------------

data = np.load(TARGETS_OUT)

token_ids = torch.tensor(data["token_ids"], dtype=torch.long)
mapped300 = torch.tensor(data["mapped"], dtype=torch.float32)  # [N,300]

# -------------------------------------------------
# Get BERT embeddings (target)
# -------------------------------------------------

with torch.no_grad():
    emb = model_base.get_input_embeddings().weight.detach().cpu()  # [V,768]

target768 = emb[token_ids]  # [N,768]

print("Pairs:", mapped300.shape, target768.shape)

# -------------------------------------------------
# Projection model
# -------------------------------------------------

proj = nn.Linear(300, 768).to(device)

opt = torch.optim.AdamW(proj.parameters(), lr=2e-4)
loss_fn = nn.MSELoss()

# -------------------------------------------------
# Training
# -------------------------------------------------

BATCH = 2048
EPOCHS = 2

proj.train()

for ep in range(EPOCHS):

    perm = torch.randperm(mapped300.size(0))

    total_loss = 0.0
    steps = 0

    for i in range(0, mapped300.size(0), BATCH):

        idx = perm[i:i+BATCH]

        x = mapped300[idx].to(device)   # [b,300]
        y = target768[idx].to(device)   # [b,768]

        pred = proj(x)

        loss = loss_fn(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        steps += 1

    print(f"Epoch {ep+1}/{EPOCHS} | proj MSE loss: {total_loss/steps:.6f}")

# -------------------------------------------------
# Save
# -------------------------------------------------

torch.save(proj.state_dict(), PROJ_OUT)

print("Saved projection to:", PROJ_OUT)

Train LoRA + MUSE auxiliary loss (MLM + λ*align)

In [ ]:
# =========================
# CELL 5 — Train LoRA on BASE BERT + MUSE auxiliary loss
# =========================

import os, math
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    BertTokenizer, BertForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ use uncased English BERT consistently
BASE_MODEL = "bert-base-uncased"

TRAIN_FILE = "/content/drive/MyDrive/SLA_Project_CSEE/splits/incorrect_train.txt"
TEST_FILE  = "/content/drive/MyDrive/SLA_Project_CSEE/splits/incorrect_test.txt"

# ✅ renamed output path for your uncased-BERT setup
ADAPTER_OUT = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/lora_incorrect_base_bert_muse"
os.makedirs(ADAPTER_OUT, exist_ok=True)

TARGETS_OUT = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/tokenid_to_mapped300.npz"
PROJ_OUT = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/proj_300_to_768.pt"

# ---- hyperparams
MAX_LEN = 64
BATCH = 16
EPOCHS = 3
LR = 2e-4
LAMBDA_ALIGN = 0.05
ALIGN_SAMPLE_K = 256

# load text
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_lines = [x.strip() for x in f if x.strip()]

print("Train lines:", len(train_lines))

ds_train = Dataset.from_dict({"text": train_lines})

tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

def tok_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

ds_train = ds_train.map(tok_fn, batched=True, remove_columns=["text"])
ds_train.set_format(type="torch", columns=["input_ids", "attention_mask"])

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# ---- base model + LoRA
base = BertForMaskedLM.from_pretrained(BASE_MODEL)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["query", "value"],
    bias="none",
    task_type="CAUSAL_LM"   # PEFT supports this for BERT LoRA setup
)

model = get_peft_model(base, lora_cfg)
model.to(device)
model.print_trainable_parameters()

# ---- load target table (token_id -> mapped300) and projection
npz = np.load(TARGETS_OUT)
token_ids = torch.tensor(npz["token_ids"], dtype=torch.long)   # [N]
mapped300 = torch.tensor(npz["mapped"], dtype=torch.float32)   # [N,300]

id_to_index = {int(t.item()): i for i, t in enumerate(token_ids)}

proj = nn.Linear(300, 768)
proj.load_state_dict(torch.load(PROJ_OUT, map_location="cpu"))
proj = proj.to(device)
proj.eval()

# precompute mapped -> projected BERT-space targets
with torch.no_grad():
    target_proj768 = proj(mapped300.to(device)).detach()   # [N,768]

print("target_proj768:", target_proj768.shape, "device:", target_proj768.device)

# ---- custom Trainer: add align loss
class LoraMuseTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        mlm_loss = outputs.loss

        input_ids = inputs["input_ids"]  # [B,L]

        # ✅ simpler and safer for PEFT-wrapped model
        emb_layer = model.get_input_embeddings()
        emb = emb_layer(input_ids)       # [B,L,768]

        flat_ids = input_ids.view(-1)                     # [B*L]
        flat_emb = emb.view(-1, emb.size(-1))            # [B*L,768]

        alignable = []
        for j, tid in enumerate(flat_ids.tolist()):
            if tid in id_to_index:
                alignable.append((j, id_to_index[tid]))

        if len(alignable) == 0:
            total_loss = mlm_loss
            return (total_loss, outputs) if return_outputs else total_loss

        if len(alignable) > ALIGN_SAMPLE_K:
            picked = torch.randperm(len(alignable))[:ALIGN_SAMPLE_K].tolist()
            alignable = [alignable[i] for i in picked]

        j_idx = torch.tensor([a[0] for a in alignable], device=device)
        t_idx = torch.tensor([a[1] for a in alignable], device=device)

        pred_vec = flat_emb[j_idx]          # [k,768]
        tgt_vec = target_proj768[t_idx]     # [k,768]

        pred_norm = pred_vec / (pred_vec.norm(dim=1, keepdim=True) + 1e-8)
        tgt_norm  = tgt_vec  / (tgt_vec.norm(dim=1, keepdim=True) + 1e-8)

        align_loss = 1.0 - (pred_norm * tgt_norm).sum(dim=1).mean()

        total_loss = mlm_loss + LAMBDA_ALIGN * align_loss

        if return_outputs:
            outputs.align_loss = align_loss.detach()
            outputs.mlm_loss = mlm_loss.detach()
            return total_loss, outputs

        return total_loss

args = TrainingArguments(
    output_dir=ADAPTER_OUT,
    per_device_train_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    save_strategy="epoch",
    logging_steps=200,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = LoraMuseTrainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    data_collator=collator,
)

print("Training LoRA + MUSE auxiliary alignment...")
trainer.train()

# Save adapter
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

print("Saved LoRA+MUSE adapter to:", ADAPTER_OUT)

In [ ]:
from transformers import BertForMaskedLM
from peft import PeftModel

# ✅ define your adapter path (IMPORTANT)
ADAPTER_PATH = "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/lora_incorrect_base_bert_muse"

# load base model
base = BertForMaskedLM.from_pretrained("bert-base-uncased")

# attach LoRA adapter
model = PeftModel.from_pretrained(base, ADAPTER_PATH)

model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

print("Model loaded successfully ✅")

Evaluate perplexity (incorrect test + mixed test)

In [ ]:
# ============================================
# Evaluate BERT Models (Pseudo Perplexity)
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import BertForMaskedLM, BertTokenizer
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# Tokenizer
# --------------------------------------------

BASE_MODEL = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

# --------------------------------------------
# Pseudo-Perplexity Function
# --------------------------------------------

def compute_pseudo_perplexity(text, tokenizer, model, max_length=64):

    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():

        for i in range(1, seq_len - 1):

            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)

            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    ppl = math.exp(-log_likelihood / token_count)
    return ppl


# --------------------------------------------
# Models (UPDATED: include LoRA + MUSE)
# --------------------------------------------

models_config = {

    "Base_BERT": "bert-base-uncased",

    "LoRA_Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora/lora_incorrect_base_bert",

    "LoRA_MUSE_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/lora_incorrect_base_bert_muse"
}


# --------------------------------------------
# Evaluation Datasets
# --------------------------------------------

essay_files = {
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# Loop Over Models
# ============================================

for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading model:", model_name)
    print("========================================")

    # 🔹 Base BERT
    if model_name == "Base_BERT":
        model = BertForMaskedLM.from_pretrained(model_path).to(device)

    # 🔹 LoRA models (both normal + MUSE)
    else:
        base_model = BertForMaskedLM.from_pretrained(BASE_MODEL)
        model = PeftModel.from_pretrained(base_model, model_path)
        model = model.to(device)

    model.eval()

    seed_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        if group == "Chinese_ICNALE":

            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:

            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):

            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        # ✅ avoid crash if empty
        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")

        seed_results[group] = avg_ppl

        print(f"{group} Average PPL: {avg_ppl:.3f}")

    all_results[model_name] = seed_results


# ============================================
# Final Results
# ============================================

print("\n\nFINAL RESULTS")
print("========================================")

for group in essay_files.keys():

    print(f"\n{group}")

    for model_name in models_config.keys():

        value = all_results[model_name][group]

        if np.isnan(value):
            print(f"{model_name}: NaN")
        else:
            print(f"{model_name}: {value:.3f}")


In [ ]:
# Full ONE-CELL code for LoRA + MUSE BERT model

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, BertForMaskedLM
from peft import PeftModel

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK_FIXED.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ✅ UPDATED: LoRA + MUSE MODEL
MODEL_DIRS = [
    "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/lora_incorrect_base_bert_muse"
]

BASE_MODEL = "bert-base-uncased"

OPTIONS = [
    "on","at","like","as","with","inside","of","among","in",
    "by","from","during","about","near","out","round","until",
    "along","for","against","across","to","off","upon","towards",
    "under","around","behind","besides","within","beside","into",
    "between","up","over","before","above","down","after","till",
    "toward","without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES
# -------------------------------------------------

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        if "[MASK]" not in line:
            parts = line.split()

            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# LOAD OPTION LINES
# -------------------------------------------------

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOOP OVER MODELS
# -------------------------------------------------

for model_dir in MODEL_DIRS:

    model_name = os.path.basename(model_dir.rstrip("/"))

    print("\n========================================")
    print("Running model:", model_name)
    print("Model path:", model_dir)
    print("========================================")

    # ✅ tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

    # ✅ base + LoRA load (IMPORTANT)
    base_model = BertForMaskedLM.from_pretrained(BASE_MODEL)
    model = PeftModel.from_pretrained(base_model, model_dir)

    model.to(device)
    model.eval()

    # -------------------------------------------------
    # Option token ids (handle missing tokens safely)
    # -------------------------------------------------

    option_token_ids = {}
    for w in OPTIONS:
        tid = tokenizer.convert_tokens_to_ids(w)
        option_token_ids[w] = tid if tid is not None else -1

    mask_token = tokenizer.mask_token
    mask_token_id = tokenizer.mask_token_id

    if mask_token is None:
        raise ValueError("Tokenizer missing mask token")

    # -------------------------------------------------
    # MASK PREDICTION FUNCTION
    # -------------------------------------------------

    def predict_masked_word(sentence):

        sent = sentence.replace("[MASK]", mask_token)

        inputs = tokenizer(sent, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)

        if mask_positions.numel() == 0:
            return {w: 0.0 for w in OPTIONS}

        token_idx = int(mask_positions[0, 1].item())

        with torch.no_grad():
            logits = model(**inputs).logits

        mask_logits = logits[0, token_idx, :]
        probs = torch.softmax(mask_logits, dim=-1)

        word_probs = {}

        for w in OPTIONS:
            wid = option_token_ids[w]

            if wid < 0 or wid >= probs.size(-1):
                word_probs[w] = 0.0
            else:
                word_probs[w] = float(probs[wid].item())

        return word_probs

    # -------------------------------------------------
    # RAW PREDICTIONS
    # -------------------------------------------------

    rows = []

    for s in sentences:
        probs = predict_masked_word(s)

        row = {"Question": s}
        row.update(probs)

        rows.append(row)

    df_raw = pd.DataFrame(rows)

    raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
    df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")

    print("Raw predictions saved to:", raw_path)

    # -------------------------------------------------
    # NORMALIZED SCORES
    # -------------------------------------------------

    normalized_rows = []

    for i, allowed in enumerate(option_lines):

        allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]
        sent = sentences[i]

        total = sum(float(df_raw.loc[i, w]) for w in allowed_list if w in df_raw.columns)

        row = {"sentence": sent}

        for w in OPTIONS:
            if (w in allowed_list) and (total > 0):
                row[w] = float(df_raw.loc[i, w]) / total
            else:
                row[w] = 0.0

        normalized_rows.append(row)

    df_norm = pd.DataFrame(normalized_rows)

    norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
    df_norm.to_csv(norm_path, index=False, encoding="utf-8-sig")

    print("Normalized scores saved to:", norm_path)

    del model
    torch.cuda.empty_cache()

print("\nALL MODELS FINISHED.")

In [ ]:
# ============================================
# FINAL — Compare ALL MODELS (Pseudo Perplexity)
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import BertForMaskedLM, BertTokenizer
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# Tokenizer (same for all)
# --------------------------------------------

BASE_MODEL = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

# --------------------------------------------
# Pseudo-Perplexity
# --------------------------------------------

def compute_pseudo_perplexity(text, tokenizer, model, max_length=64):

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):

            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)

# --------------------------------------------
# ALL MODELS (FINAL SET)
# --------------------------------------------

models_config = {
    "Base_BERT": "bert-base-uncased",
    "Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models/csee_incorrect_bert",
    "LoRA_Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora/lora_incorrect_base_bert",
    "LoRA_MUSE_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/lora_incorrect_base_bert_muse"
}

# --------------------------------------------
# DATASETS
# --------------------------------------------

essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# LOOP OVER MODELS
# ============================================

for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading:", model_name)
    print("========================================")

    # Base or fine-tuned
    if model_name in ["Base_BERT", "Incorrect_BERT"]:
        model = BertForMaskedLM.from_pretrained(model_path).to(device)

    # LoRA models
    else:
        base_model = BertForMaskedLM.from_pretrained(BASE_MODEL)
        model = PeftModel.from_pretrained(base_model, model_path)
        model = model.to(device)

    model.eval()

    results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        # Special Chinese split
        if group == "Chinese":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]
        else:
            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):
            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")

        results[group] = avg_ppl

        print(f"{group}: {avg_ppl:.3f}")

    all_results[model_name] = results

# ============================================
# FINAL TABLE
# ============================================

print("\n\nFINAL COMPARISON TABLE")
print("========================================")

groups = list(essay_files.keys())

# header
print(f"{'Model':<25}", end="")
for g in groups:
    print(f"{g:<12}", end="")
print()

# rows
for model_name in models_config.keys():
    print(f"{model_name:<25}", end="")

    for g in groups:
        val = all_results[model_name][g]

        if np.isnan(val):
            print(f"{'NaN':<12}", end="")
        else:
            print(f"{val:.2f}".ljust(12), end="")

    print()

Training loop (CLEAN — no Trainer needed)

In [ ]:
!pip install gensim -q

In [ ]:
import os
import torch
import numpy as np
from pathlib import Path
from transformers import BertTokenizer, BertForMaskedLM, TrainingArguments
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import KeyedVectors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
import os
from gensim.models import KeyedVectors

VEC_PATH = "/content/drive/MyDrive/cc.zh.300.vec"
GZ_PATH = VEC_PATH + ".gz"

print("Loading FastText...", flush=True)

# -------------------------------------------------
# STEP 1 — Check if .vec exists
# -------------------------------------------------

if not os.path.exists(VEC_PATH):

    print(" .vec file not found")

    # -------------------------------------------------
    # STEP 2 — Check if .gz exists
    # -------------------------------------------------
    if os.path.exists(GZ_PATH):
        print("📦 Found .gz file → Unzipping...", flush=True)
        !gunzip "{GZ_PATH}"

    else:
        print("⬇ Downloading FastText (this may take time)...", flush=True)

        !wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.zh.300.vec.gz
        !mv cc.zh.300.vec.gz /content/drive/MyDrive/

        print("📦 Unzipping...", flush=True)
        !gunzip "/content/drive/MyDrive/cc.zh.300.vec.gz"

# -------------------------------------------------
# STEP 3 — Load vectors
# -------------------------------------------------

print(" Loading vectors into memory (1–3 min)...", flush=True)

zh_vectors = KeyedVectors.load_word2vec_format(
    VEC_PATH,
    binary=False
)

print(" FastText loaded successfully")
print("Vocab size:", len(zh_vectors))

# -------------------------------------------------
# STEP 4 — Save optimized version (VERY IMPORTANT)
# -------------------------------------------------

MODEL_SAVE_PATH = "/content/drive/MyDrive/cc.zh.300.model"

if not os.path.exists(MODEL_SAVE_PATH):
    print("💾 Saving optimized version...", flush=True)
    zh_vectors.save(MODEL_SAVE_PATH)
    print("Saved at:", MODEL_SAVE_PATH)
else:
    print("⚡ Optimized model already exists → use next time for faster loading")

In [ ]:
import torch
import pandas as pd
import os
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = Path("/content/drive/MyDrive/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
def load_bilingual_dict(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            zh, en = line.strip().split()
            pairs.append((zh, en))
    return pairs

DICT_PATH = "/content/drive/MyDrive/MUSE/MUSE/data/dictionaries/zh-en.txt"

bilingual_pairs = load_bilingual_dict(DICT_PATH)

print("Pairs loaded:", len(bilingual_pairs))

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
from torch.utils.data import Dataset

class MUSEOnlyDataset(Dataset):

    def __init__(self, pairs, zh_vectors, tokenizer, max_len=8):
        self.data = []

        print("Building dataset...", flush=True)

        for i, (zh_word, _) in enumerate(pairs):

            if zh_word not in zh_vectors:
                continue

            if i % 2000 == 0:
                print(f"Processed {i}", flush=True)

            encoding = tokenizer(
                zh_word,   # ✅ FIXED (no char split)
                padding="max_length",
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            )

            vec = torch.tensor(zh_vectors[zh_word], dtype=torch.float32)

            self.data.append({
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
                "fasttext": vec
            })

        print("Final dataset size:", len(self.data))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
from torch.utils.data import DataLoader

dataset = MUSEOnlyDataset(bilingual_pairs, zh_vectors, tokenizer)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
from transformers import BertForMaskedLM
import torch.nn as nn

model = BertForMaskedLM.from_pretrained("bert-base-uncased")

VECTOR_DIM = 300

model.muse_projection = nn.Linear(
    model.config.hidden_size,
    VECTOR_DIM,
    bias=False
)

model = model.to(device)

print("Model ready")

In [ ]:
import torch.nn.functional as F

def train_muse(model, loader, epochs=3):

    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

    model.train()

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}")

        total_loss = 0

        for i, batch in enumerate(loader):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            target_vec = batch["fasttext"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )

            hidden = outputs.hidden_states[-1][:, 0, :]
            pred_vec = model.muse_projection(hidden)

            loss = F.mse_loss(pred_vec, target_vec)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if i % 200 == 0:
                print(f"Step {i} Loss: {loss.item():.4f}", flush=True)

        print("Epoch Loss:", total_loss / len(loader))

In [ ]:
train_muse(model, loader, epochs=3)

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/outputs/muse_only_uncased"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Saved to:", SAVE_PATH)

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM

MODEL_DIR = "/content/drive/MyDrive/outputs/muse_only_uncased"

tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model = BertForMaskedLM.from_pretrained(MODEL_DIR)

model.eval()

print("Model loaded")

In [ ]:
# ============================================
# FULL ONE-CELL — MUSE FULL MODEL (NOT LoRA)
# ============================================

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, BertForMaskedLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

INPUT_FILE = "/content/drive/MyDrive/Mandarin_Question_MASK_FIXED.txt"
OPTIONS_FILE = "/content/drive/MyDrive/Mandarin_answers.txt"

OUTPUT_DIR = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ✅ YOUR FINAL TRAINED MODEL (FULL MODEL, NOT LoRA)
MODEL_DIR = "/content/drive/MyDrive/outputs/muse_only_uncased"

OPTIONS = [
    "on","at","like","as","with","inside","of","among","in",
    "by","from","during","about","near","out","round","until",
    "along","for","against","across","to","off","upon","towards",
    "under","around","behind","besides","within","beside","into",
    "between","up","over","before","above","down","after","till",
    "toward","without"
]

# -------------------------------------------------
# DEVICE
# -------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# LOAD SENTENCES
# -------------------------------------------------

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        if "[MASK]" not in line:
            parts = line.split()

            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# LOAD OPTION LINES
# -------------------------------------------------

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError(
        f"OPTIONS_FILE line count ({len(option_lines)}) must match INPUT_FILE line count ({len(sentences)})."
    )

# -------------------------------------------------
# LOAD MODEL (FIXED — FULL MODEL)
# -------------------------------------------------

print("\nLoading FULL MUSE model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = BertForMaskedLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

print("Model loaded successfully ✅")

# -------------------------------------------------
# OPTION TOKEN IDS
# -------------------------------------------------

option_token_ids = {}
for w in OPTIONS:
    tid = tokenizer.convert_tokens_to_ids(w)
    option_token_ids[w] = tid if tid is not None else -1

mask_token = tokenizer.mask_token
mask_token_id = tokenizer.mask_token_id

if mask_token is None:
    raise ValueError("Tokenizer missing mask token")

# -------------------------------------------------
# MASK PREDICTION FUNCTION
# -------------------------------------------------

def predict_masked_word(sentence):

    sent = sentence.replace("[MASK]", mask_token)

    inputs = tokenizer(sent, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    mask_positions = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=False)

    if mask_positions.numel() == 0:
        return {w: 0.0 for w in OPTIONS}

    token_idx = int(mask_positions[0, 1].item())

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[0, token_idx, :]
    probs = torch.softmax(mask_logits, dim=-1)

    word_probs = {}

    for w in OPTIONS:
        wid = option_token_ids[w]

        if wid < 0 or wid >= probs.size(-1):
            word_probs[w] = 0.0
        else:
            word_probs[w] = float(probs[wid].item())

    return word_probs

# -------------------------------------------------
# RAW PREDICTIONS
# -------------------------------------------------

print("\nRunning predictions...")

rows = []

for s in sentences:
    probs = predict_masked_word(s)

    row = {"Question": s}
    row.update(probs)

    rows.append(row)

df_raw = pd.DataFrame(rows)

model_name = "muse_only_bert"

raw_path = f"{OUTPUT_DIR}/{model_name}_raw.csv"
df_raw.to_csv(raw_path, index=False, encoding="utf-8-sig")

print("Raw predictions saved to:", raw_path)

# -------------------------------------------------
# NORMALIZED SCORES
# -------------------------------------------------

normalized_rows = []

for i, allowed in enumerate(option_lines):

    allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]
    sent = sentences[i]

    total = sum(float(df_raw.loc[i, w]) for w in allowed_list if w in df_raw.columns)

    row = {"sentence": sent}

    for w in OPTIONS:
        if (w in allowed_list) and (total > 0):
            row[w] = float(df_raw.loc[i, w]) / total
        else:
            row[w] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(normalized_rows)

norm_path = f"{OUTPUT_DIR}/{model_name}_normalized.csv"
df_norm.to_csv(norm_path, index=False, encoding="utf-8-sig")

print("Normalized scores saved to:", norm_path)

print("\nALL DONE ")

In [ ]:
# ============================================
# FINAL — Compare ALL MODELS (Pseudo Perplexity)
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import BertForMaskedLM, BertTokenizer
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# Tokenizer (same for all)
# --------------------------------------------

BASE_MODEL = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)

# --------------------------------------------
# Pseudo-Perplexity
# --------------------------------------------

def compute_pseudo_perplexity(text, tokenizer, model, max_length=64):

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):

            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)

# --------------------------------------------
# ALL MODELS (FINAL SET)
# --------------------------------------------

models_config = {
    "Base_BERT": "bert-base-uncased",
    "Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models/csee_incorrect_bert",
    "LoRA_Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora/lora_incorrect_base_bert",
    "LoRA_MUSE_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/lora_incorrect_base_bert_muse",
    "MUSE_ONLY_BERT": "/content/drive/MyDrive/outputs/muse_only_uncased"
}

# --------------------------------------------
# DATASETS
# --------------------------------------------

essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# LOOP OVER MODELS
# ============================================

# ============================================
# LOOP OVER MODELS
# ============================================

for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading:", model_name)
    print("========================================")

    # Base or normal fine-tuned models
    if model_name in ["Base_BERT", "Incorrect_BERT", "MUSE_ONLY_BERT"]:
        model = BertForMaskedLM.from_pretrained(model_path).to(device)

    #  LoRA models
    elif "LoRA" in model_name:
        base_model = BertForMaskedLM.from_pretrained(BASE_MODEL)
        model = PeftModel.from_pretrained(base_model, model_path)
        model = model.to(device)

    model.eval()

    results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        # Special Chinese split
        if group == "Chinese":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]
        else:
            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):
            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            ppl = compute_pseudo_perplexity(essay, tokenizer, model)

            if ppl is not None:
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")

        results[group] = avg_ppl

        print(f"{group}: {avg_ppl:.3f}")

    all_results[model_name] = results

# ============================================
# FINAL TABLE
# ============================================

print("\n\nFINAL COMPARISON TABLE")
print("========================================")

groups = list(essay_files.keys())

# header
print(f"{'Model':<25}", end="")
for g in groups:
    print(f"{g:<12}", end="")
print()

# rows
for model_name in models_config.keys():
    print(f"{model_name:<25}", end="")

    for g in groups:
        val = all_results[model_name][g]

        if np.isnan(val):
            print(f"{'NaN':<12}", end="")
        else:
            print(f"{val:.2f}".ljust(12), end="")

    print()

In [ ]:
with open("/content/drive/MyDrive/Mandarin_answers.txt", "r") as f:
    option_lines = [line.strip() for line in f if line.strip()]

In [ ]:
MODEL_PATHS = {
    # BASE bert as BASE LINE
   "Base_BERT": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/Base_BERT_normalized.csv",
    # tuned BERT(uncased) models with differnt techinques MUSE auxilory loss oR LORA or on just essays written by chinese students or
    "MUSE_ONLY": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/muse_only_bert_normalized.csv",
    "LoRA_MUSE": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/lora_incorrect_base_bert_muse_normalized.csv",
    "LoRA": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/lora_incorrect_bert_normalized.csv",
    "CSEE": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/csee_incorrect_bert_normalized.csv",
    "BERT_CSEE_FT": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/bert_csee_ft_normalized.csv",

    # -----------------------------
    # BabyRoBERTa (NON-MIXED)
    # -----------------------------
    "BabyRoBERTa_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed42_final_normalized.csv",
    "BabyRoBERTa_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed97_final_normalized.csv",
    "BabyRoBERTa_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed123_final_normalized.csv",
    "BabyRoBERTa_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed420_final_normalized.csv",
    "BabyRoBERTa_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed999_final_normalized.csv",

    # -----------------------------
    # BabyRoBERTa (MIXED)
    # -----------------------------
    "BabyRoBERTa_MIX_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed42_final_normalized.csv",
    "BabyRoBERTa_MIX_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed97_final_normalized.csv",
    "BabyRoBERTa_MIX_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed123_final_normalized.csv",
    "BabyRoBERTa_MIX_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed420_final_normalized.csv",
    "BabyRoBERTa_MIX_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed999_final_normalized.csv",
     # BabyRoberta (Mixed 30 + 5)

    "BabyRoBERTa_30M+5M_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed42_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed97_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed123_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed420_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed999_final_normalized.csv",
    # BabyRoBERTa (25M + 5M Freeze)

    "BabyRoBERTa_25M+5M_freeze_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed42_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed97_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed123_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed420_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed999_normalized.csv",

}


In [ ]:
import os

for name, path in MODEL_PATHS.items():
    print(name, "→", os.path.exists(path))

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Mandarin_SLA_Project_BABYROBERTA/babyroberta_normalized_scores.csv")
print(df.head())

In [ ]:
gold = pd.read_csv("/content/drive/MyDrive/gold.csv")
print(gold.head())

In [ ]:
import re

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

In [ ]:
import pandas as pd

def compute_accuracy_correct_only(gold_path, model_path):

    gold = pd.read_csv(gold_path)
    model = pd.read_csv(model_path)

    gold["question_norm"] = gold["question"].apply(normalize_text)
    model["sentence_norm"] = model["sentence"].apply(normalize_text)

    gold["Precent"] = pd.to_numeric(gold["Precent"], errors="coerce")

    # 👉 ONLY PERFECT HUMAN AGREEMENT
    gold = gold[gold["Precent"] >= 0.99]

    print("Total correct-only questions:", gold["question_norm"].nunique())

    common = set(gold["question_norm"]).intersection(set(model["sentence_norm"]))

    correct = 0
    total = 0

    grouped = gold.groupby("question_norm")

    for question, group in grouped:

        if question not in common:
            continue

        match = model[model["sentence_norm"] == question]

        if len(match) == 0:
            continue

        gold_answer = group.iloc[0]["preposition"]

        model_row = match.iloc[0]

        prob_cols = [c for c in model.columns if c not in ["sentence", "sentence_norm"]]
        probs = model_row[prob_cols]

        if probs.isnull().any():
            continue

        model_answer = probs.idxmax()

        if model_answer == gold_answer:
            correct += 1

        total += 1

    acc = correct / total if total > 0 else 0

    print(f"Accuracy (correct-only TRUE): {acc:.4f} ({correct}/{total})")
    return acc

In [ ]:
gold_path = "/content/drive/MyDrive/gold.csv"

model_path = "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed42_final_normalized.csv"

compute_accuracy_strict(gold_path, model_path)

In [ ]:
for name, path in MODEL_PATHS.items():
    print("\nModel:", name)
    compute_accuracy_strict("/content/drive/MyDrive/gold.csv", path)

In [ ]:
results = {}

for name, path in MODEL_PATHS.items():
    print("\nModel:", name)
    acc = compute_accuracy_strict("/content/drive/MyDrive/gold.csv", path)
    results[name] = acc

In [ ]:
import numpy as np

def print_final_results(results):

    print("\n===== FINAL MODEL RESULTS =====\n")

    # -----------------------------
    # BASELINES
    # -----------------------------
    baseline_models = [
        "Base_BERT",
        "MUSE_ONLY",
        "LoRA_MUSE",
        "LoRA",
        "CSEE",
        "BERT_CSEE_FT",
    ]

    print("----- BASELINES -----")
    for m in baseline_models:
        if m in results:
            print(f"{m}: {results[m]:.4f}")

    # -----------------------------
    # BABYROBERTA NON-MIXED
    # -----------------------------
    print("\n----- BabyRoBERTa (Non-Mixed Seeds) -----")

    babyberta = []
    for seed in ["42", "97", "123", "420", "999"]:
        key = f"BabyRoBERTa_seed{seed}"
        if key in results:
            val = results[key]
            babyberta.append(val)
            print(f"{key}: {val:.4f}")

    # -----------------------------
    # BABYROBERTA MIXED (25+5)
    # -----------------------------
    print("\n----- BabyRoBERTa (Mixed 25+5 Seeds) -----")

    babyberta_mix = []
    for seed in ["42", "97", "123", "420", "999"]:
        key = f"BabyRoBERTa_MIX_seed{seed}"
        if key in results:
            val = results[key]
            babyberta_mix.append(val)
            print(f"{key}: {val:.4f}")

    # -----------------------------
    # BABYROBERTA 30M+5M
    # -----------------------------
    print("\n----- BabyRoBERTa (30M+5M Seeds) -----")

    babyberta_plus5 = []
    for seed in ["42", "97", "123", "420", "999"]:
        key = f"BabyRoBERTa_30M+5M_seed{seed}"
        if key in results:
            val = results[key]
            babyberta_plus5.append(val)
            print(f"{key}: {val:.4f}")

    # -----------------------------
    # 🆕 BABYROBERTA FREEZE (25M+5M)
    # -----------------------------
    print("\n----- BabyRoBERTa (25M+5M Freeze Seeds) -----")

    babyberta_freeze = []
    for seed in ["42", "97", "123", "420", "999"]:
        key = f"BabyRoBERTa_25M+5M_freeze_seed{seed}"
        if key in results:
            val = results[key]
            babyberta_freeze.append(val)
            print(f"{key}: {val:.4f}")

    # -----------------------------
    # FINAL AVERAGES
    # -----------------------------
    print("\n===== FINAL BABYROBERTA RESULTS =====")

    if len(babyberta) > 0:
        print("\nBabyRoBERTa (30M Non-Mixed)")
        print(f"Mean Accuracy: {np.mean(babyberta):.4f}")
        print(f"Std Dev      : {np.std(babyberta):.4f}")

    if len(babyberta_mix) > 0:
        print("\nBabyRoBERTa (30M Mixed 25+5)")
        print(f"Mean Accuracy: {np.mean(babyberta_mix):.4f}")
        print(f"Std Dev      : {np.std(babyberta_mix):.4f}")

    if len(babyberta_plus5) > 0:
        print("\nBabyRoBERTa (30M+5M)")
        print(f"Mean Accuracy: {np.mean(babyberta_plus5):.4f}")
        print(f"Std Dev      : {np.std(babyberta_plus5):.4f}")

    if len(babyberta_freeze) > 0:
        print("\nBabyRoBERTa (25M+5M Freeze)")
        print(f"Mean Accuracy: {np.mean(babyberta_freeze):.4f}")
        print(f"Std Dev      : {np.std(babyberta_freeze):.4f}")

In [ ]:
print_final_results(results)

In [ ]:
import pandas as pd

def compute_accuracy_hard_cases(gold_path, model_path, margin=0.05):

    gold = pd.read_csv(gold_path)
    model = pd.read_csv(model_path)

    gold["question_norm"] = gold["question"].apply(normalize_text)
    model["sentence_norm"] = model["sentence"].apply(normalize_text)

    gold["Precent"] = pd.to_numeric(gold["Precent"], errors="coerce")

    correct = 0
    total = 0

    grouped = gold.groupby("question_norm")

    for question, group in grouped:

        if question not in set(model["sentence_norm"]):
            continue

        # sort by percentage
        group = group.sort_values(by="Precent", ascending=False)

        if len(group) < 2:
            continue

        # ✅ correct = highest %
        top = group.iloc[0]
        second = group.iloc[1]

        correct_answer = top["preposition"]
        top_percent = top["Precent"]
        second_percent = second["Precent"]

        # ------------------------------------
        # ✅ HARD CASE CONDITION
        # ------------------------------------
        # if second is close to top → confusion
        if (top_percent - second_percent) > margin:
            continue

        # ------------------------------------
        # model prediction
        # ------------------------------------
        match = model[model["sentence_norm"] == question]

        if len(match) == 0:
            continue

        model_row = match.iloc[0]

        prob_cols = [c for c in model.columns if c not in ["sentence", "sentence_norm"]]
        probs = model_row[prob_cols]

        if probs.isnull().any():
            continue

        model_answer = probs.idxmax()

        if model_answer == correct_answer:
            correct += 1

        total += 1

    acc = correct / total if total > 0 else 0

    print(f"Accuracy (confusing cases): {acc:.4f} ({correct}/{total})")
    return acc

In [ ]:
results_hard = {}

gold_path = "/content/drive/MyDrive/gold.csv"

for name, path in MODEL_PATHS.items():
    print("\n========================================")
    print("Model:", name)

    acc = compute_accuracy_hard_cases(gold_path, path, margin=0.05)
    results_hard[name] = acc

In [ ]:
gold = pd.read_excel("/content/drive/MyDrive/mandarin_Questions_Full.xlsx")
print(gold.columns)

In [ ]:
gold = pd.read_excel("/content/drive/MyDrive/mandarin_Questions_Full.xlsx")
model = pd.read_csv(MODEL_PATHS["Base_BERT"])  # just for test

# ✅ MUST create these again
gold["question_norm"] = gold["Question"].apply(normalize_text)
model["sentence_norm"] = model["sentence"].apply(normalize_text)

In [ ]:
MODEL_PATHS = {
    # BASE bert as BASE LINE
   "Base_BERT": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/Base_BERT_normalized.csv",
    # tuned BERT(uncased) models with differnt techinques MUSE auxilory loss oR LORA or on just essays written by chinese students or
    "MUSE_ONLY": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/muse_only_bert_normalized.csv",
    "LoRA_MUSE": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/lora_incorrect_base_bert_muse_normalized.csv",
    "LoRA": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/lora_incorrect_bert_normalized.csv",
    "CSEE": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/csee_incorrect_bert_normalized.csv",
    "BERT_CSEE_FT": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/bert_csee_ft_normalized.csv",

    # -----------------------------
    # BabyRoBERTa (NON-MIXED)
    # -----------------------------
    "BabyRoBERTa_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed42_final_normalized.csv",
    "BabyRoBERTa_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed97_final_normalized.csv",
    "BabyRoBERTa_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed123_final_normalized.csv",
    "BabyRoBERTa_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed420_final_normalized.csv",
    "BabyRoBERTa_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_seed999_final_normalized.csv",

    # -----------------------------
    # BabyRoBERTa (MIXED)
    # -----------------------------
    "BabyRoBERTa_MIX_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed42_final_normalized.csv",
    "BabyRoBERTa_MIX_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed97_final_normalized.csv",
    "BabyRoBERTa_MIX_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed123_final_normalized.csv",
    "BabyRoBERTa_MIX_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed420_final_normalized.csv",
    "BabyRoBERTa_MIX_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_mixed_seed999_final_normalized.csv",
     # BabyRoberta (Mixed 30 + 5)

    "BabyRoBERTa_30M+5M_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed42_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed97_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed123_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed420_final_normalized.csv",
    "BabyRoBERTa_30M+5M_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_30M_plus5M_seed999_final_normalized.csv",
    # BabyRoBERTa (25M + 5M Freeze)

    "BabyRoBERTa_25M+5M_freeze_seed42": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed42_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed97": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed97_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed123": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed123_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed420": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed420_normalized.csv",
    "BabyRoBERTa_25M+5M_freeze_seed999": "/content/drive/MyDrive/Mandarin_SLA_Project_CSEE/babyroberta_25M_then_5M_freeze_seed999_normalized.csv",

}

In [ ]:
import pandas as pd

path = MODEL_PATHS["Base_BERT"]

df = pd.read_csv(path)

print("COLUMNS:")
print(df.columns)

print("\nSAMPLE ROWS:")
print(df.head(5))

print("\nFIRST SENTENCE:")
print(df["sentence"].iloc[0])

In [ ]:
gold = pd.read_excel("/content/drive/MyDrive/mandarin_Questions_Full.xlsx")

print("COLUMNS:")
print(gold.columns)

print("\nSAMPLE ROWS:")
print(gold.head(5))

print("\nFIRST QUESTION:")
print(gold["Question"].iloc[0])

In [ ]:
gold = pd.read_excel("/content/drive/MyDrive/mandarin_Questions_Full.xlsx")
model = pd.read_csv(MODEL_PATHS["Base_BERT"])

# ✅ create normalized columns
gold["question_norm"] = gold["Question.1"].apply(normalize_text)
model["sentence_norm"] = model["sentence"].apply(normalize_text)

In [ ]:
matched = 0

for q in gold["question_norm"].unique():
    if any(q in s or s in q for s in model["sentence_norm"]):
        matched += 1

print("Matched:", matched, "/", len(gold["question_norm"].unique()))

In [ ]:
import pandas as pd
import re

# --------------------------------------------
# NORMALIZATION (STRONG)
# --------------------------------------------
def normalize_text(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    # unify blanks
    text = text.replace("____", "[mask]")

    # remove punctuation (except words)
    text = re.sub(r"[^\w\s]", "", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# --------------------------------------------
# 🔥 TOKEN OVERLAP MATCH (BEST APPROACH)
# --------------------------------------------
def find_best_match(question, model_df):

    q_words = set(question.split())

    best_row = None
    best_score = 0

    for _, row in model_df.iterrows():
        sent = row["sentence_norm"]
        s_words = set(sent.split())

        # overlap score
        overlap = len(q_words.intersection(s_words))

        if overlap > best_score:
            best_score = overlap
            best_row = row

    # require reasonable match
    if best_score >= 6:
        return best_row

    return None


# --------------------------------------------
# MAIN FUNCTION (CORRECT + ROBUST)
# --------------------------------------------
def compute_accuracy_wrong_dominates_excel(gold_path, model_path):

    gold = pd.read_excel(gold_path)
    model = pd.read_csv(model_path)

    QUESTION_COL = "Question.1"
    PREP_COL = "Preposition"
    PERCENT_COL = "Precent"
    CORRECT_COL = "Correct"

    # normalize
    gold["question_norm"] = gold[QUESTION_COL].apply(normalize_text)
    model["sentence_norm"] = model["sentence"].apply(normalize_text)

    gold[PERCENT_COL] = pd.to_numeric(gold[PERCENT_COL], errors="coerce")

    correct = 0
    total = 0

    total_questions = 0
    wrong_dominant_cases = 0
    matched_cases = 0

    grouped = gold.groupby("question_norm")

    for question, group in grouped:

        total_questions += 1
        group = group.sort_values(by=PERCENT_COL, ascending=False)

        if len(group) < 2:
            continue

        top = group.iloc[0]

        # professor condition
        if top[CORRECT_COL] != 0:
            continue

        wrong_dominant_cases += 1

        # match
        model_row = find_best_match(question, model)

        if model_row is None:
            continue

        matched_cases += 1

        # true correct answer
        correct_rows = group[group[CORRECT_COL] == 1]

        if len(correct_rows) == 0:
            continue

        true_answer = correct_rows.iloc[0][PREP_COL]

        # model prediction
        prob_cols = [c for c in model.columns if c not in ["sentence", "sentence_norm"]]
        probs = model_row[prob_cols]

        if probs.isnull().any():
            continue

        model_answer = probs.idxmax()

        if model_answer == true_answer:
            correct += 1

        total += 1

    acc = correct / total if total > 0 else 0

    print("\n========== FINAL SUMMARY ==========")
    print("Total questions:", total_questions)
    print("Wrong-dominant cases:", wrong_dominant_cases)
    print("Matched cases:", matched_cases)
    print("Used for evaluation:", total)
    print(f"\nAccuracy (wrong-dominates TRUE): {acc:.4f} ({correct}/{total})")

    return acc


# --------------------------------------------
# RUN
# --------------------------------------------
results_wrong = {}

gold_path = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

for name, path in MODEL_PATHS.items():
    print("\n\n########################################")
    print("Model:", name)

    acc = compute_accuracy_wrong_dominates_excel(gold_path, path)
    results_wrong[name] = acc

In [ ]:
import numpy as np

def summarize_results(results_dict):

    grouped = {
        "BabyRoBERTa (Non-Mixed)": [],
        "BabyRoBERTa (Mixed)": [],
        "BabyRoBERTa (30M+5M)": [],
        "BabyRoBERTa (25M+5M Freeze)": []
    }

    final_results = {}

    for name, acc in results_dict.items():

        if "BabyRoBERTa_seed" in name and "MIX" not in name:
            grouped["BabyRoBERTa (Non-Mixed)"].append(acc)

        elif "BabyRoBERTa_MIX" in name:
            grouped["BabyRoBERTa (Mixed)"].append(acc)

        elif "30M+5M" in name:
            grouped["BabyRoBERTa (30M+5M)"].append(acc)

        elif "25M+5M_freeze" in name:
            grouped["BabyRoBERTa (25M+5M Freeze)"].append(acc)

        else:
            # keep other models as-is
            final_results[name] = acc

    # -----------------------------------
    # compute mean + std for grouped ones
    # -----------------------------------
    for group_name, values in grouped.items():

        if len(values) > 0:
            mean = np.mean(values)
            std = np.std(values)

            final_results[group_name] = (mean, std)

    return final_results

In [ ]:
def print_summary(final_results):

    print("\n========== FINAL PAPER RESULTS ==========\n")

    for name, value in final_results.items():

        if isinstance(value, tuple):
            mean, std = value
            print(f"{name}: {mean:.4f} (±{std:.4f})")
        else:
            print(f"{name}: {value:.4f}")

In [ ]:
final_results = summarize_results(results_wrong)
print_summary(final_results)

In [ ]:
import pandas as pd
import re
import numpy as np

# --------------------------------------------
# NORMALIZATION
# --------------------------------------------
def normalize_text(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    text = text.replace("____", "[mask]")

    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# --------------------------------------------
# MATCH FUNCTION (TOKEN OVERLAP)
# --------------------------------------------
def find_best_match(question, model_df):

    q_words = set(question.split())

    best_row = None
    best_score = 0

    for _, row in model_df.iterrows():
        sent = row["sentence_norm"]
        s_words = set(sent.split())

        overlap = len(q_words.intersection(s_words))

        if overlap > best_score:
            best_score = overlap
            best_row = row

    if best_score >= 6:
        return best_row

    return None


# --------------------------------------------
# 🔥 NEW METRIC (PROFESSOR'S QUESTION)
# --------------------------------------------
def compute_human_like_error(gold_path, model_path):

    gold = pd.read_excel(gold_path)
    model = pd.read_csv(model_path)

    QUESTION_COL = "Question.1"
    PREP_COL = "Preposition"
    PERCENT_COL = "Precent"
    CORRECT_COL = "Correct"

    # normalize
    gold["question_norm"] = gold[QUESTION_COL].apply(normalize_text)
    model["sentence_norm"] = model["sentence"].apply(normalize_text)

    gold[PERCENT_COL] = pd.to_numeric(gold[PERCENT_COL], errors="coerce")

    correct = 0
    total = 0

    grouped = gold.groupby("question_norm")

    for question, group in grouped:

        group = group.sort_values(by=PERCENT_COL, ascending=False)

        if len(group) < 2:
            continue

        top = group.iloc[0]

        # 🔥 ONLY wrong-dominant cases
        if top[CORRECT_COL] != 0:
            continue

        human_wrong = top[PREP_COL]

        model_row = find_best_match(question, model)

        if model_row is None:
            continue

        prob_cols = [c for c in model.columns if c not in ["sentence", "sentence_norm"]]
        probs = model_row[prob_cols]

        if probs.isnull().any():
            continue

        model_answer = probs.idxmax()

        # ✅ match WRONG answer
        if model_answer == human_wrong:
            correct += 1

        total += 1

    acc = correct / total if total > 0 else 0

    return acc, correct, total


# --------------------------------------------
# RUN FOR ALL MODELS
# --------------------------------------------
gold_path = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

human_like_results = {}

print("\n===== HUMAN-LIKE ERROR RESULTS =====\n")

for name, path in MODEL_PATHS.items():

    acc, c, t = compute_human_like_error(gold_path, path)

    human_like_results[name] = acc

    print(f"{name}")
    print(f"  Human-like error accuracy: {acc:.4f} ({c}/{t})\n")

In [ ]:
compute_human_like_error_accuracy(
    "/content/drive/MyDrive/mandarin_Questions_Full.xlsx",
    MODEL_PATHS["Base_BERT"]
)

In [ ]:
essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

In [ ]:
import os
import re
from collections import Counter

def analyze_essay_file(path):

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # --------------------------------------------
    #  BEFORE FIX (original sentence splitting)
    # --------------------------------------------
    sentences_before = re.split(r'[.!?]', text)
    sentences_before = [s.strip() for s in sentences_before if len(s.strip()) > 0]
    num_sentences_before = len(sentences_before)

    # --------------------------------------------
    # AFTER FIX (replace newline → period)
    # --------------------------------------------
    text_fixed = text.replace("\n", ". ")

    sentences_after = re.split(r'[.!?]', text_fixed)
    sentences_after = [s.strip() for s in sentences_after if len(s.strip()) > 0]
    num_sentences_after = len(sentences_after)

    # --------------------------------------------
    # Tokenization (use fixed text for fairness)
    # --------------------------------------------
    tokens = re.findall(r'\b\w+\b', text_fixed.lower())

    num_tokens = len(tokens)
    num_types = len(set(tokens))

    avg_sentence_length = num_tokens / num_sentences_after if num_sentences_after > 0 else 0

    return {
        "sentences_before": num_sentences_before,
        "sentences_after": num_sentences_after,
        "tokens": num_tokens,
        "types": num_types,
        "avg_sentence_length": avg_sentence_length
    }


# --------------------------------------------
# YOUR FILES
# --------------------------------------------
essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}


# --------------------------------------------
# RUN ANALYSIS
# --------------------------------------------
print("\n===== CORPUS STATISTICS (Before vs After Fix) =====\n")

results = {}

for name, path in essay_files.items():

    if not os.path.exists(path):
        print(f"{name}: File not found")
        continue

    stats = analyze_essay_file(path)
    results[name] = stats

    print(f"{name}")
    print(f"  Sentences (Before fix) : {stats['sentences_before']}")
    print(f"  Sentences (After fix)  : {stats['sentences_after']}")
    print(f"  Tokens                 : {stats['tokens']}")
    print(f"  Types                  : {stats['types']}")
    print(f"  Avg Sentence Length    : {stats['avg_sentence_length']:.2f}")
    print()


In [ ]:
# ============================================
# FINAL — Compare ALL MODELS (Pseudo Perplexity)
# ============================================

import torch
import numpy as np
import re
import math
from tqdm import tqdm
from transformers import (
    BertForMaskedLM, BertTokenizer,
    RobertaForMaskedLM, RobertaTokenizer
)
from peft import PeftModel

# --------------------------------------------
# Device
# --------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# PREPOSITIONS (FULL LIST)
# --------------------------------------------

PREPOSITIONS = {
    "in","on","at","to","for","with","of","by","from",
    "about","above","across","after","against","along","among","around",
    "as","before","behind","below","beneath","beside","between","beyond",
    "during","except","inside","into","near","off","onto","out","outside",
    "over","past","since","through","throughout","toward","under","underneath",
    "until","up","upon","within","without"
}

# --------------------------------------------
# Extract sentences with prepositions (FIXED)
# --------------------------------------------

def extract_sentences_with_prepositions(text):
    sentences = re.split(r'[.!?]', text)
    valid_sentences = []

    for sent in sentences:
        words = re.findall(r'\b\w+\b', sent.lower())  # ✅ FIXED

        if any(word in PREPOSITIONS for word in words):
            if len(words) > 5:
                valid_sentences.append(sent.strip())

    return valid_sentences

# --------------------------------------------
# Tokenizers (BOTH TYPES)
# --------------------------------------------

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# --------------------------------------------
# Pseudo-Perplexity
# --------------------------------------------

# --------------------------------------------
# FAST PSEUDO-PERPLEXITY (BERT + RoBERTa SAFE)
# --------------------------------------------

def compute_pseudo_perplexity(text, tokenizer, model, max_length=64, batch_size=32):

    try:
        encodings = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        input_ids = encodings["input_ids"][0]
        attention_mask = encodings["attention_mask"][0]

        seq_len = input_ids.size(0)

        masked_inputs = []
        target_tokens = []

        # SAME LOGIC: mask each token (no change)
        for i in range(1, seq_len - 1):
            masked = input_ids.clone()
            target_tokens.append(input_ids[i].item())
            masked[i] = tokenizer.mask_token_id
            masked_inputs.append(masked)

        if len(masked_inputs) == 0:
            return None

        masked_inputs = torch.stack(masked_inputs)
        attention_mask = attention_mask.unsqueeze(0).repeat(len(masked_inputs), 1)

        masked_inputs = masked_inputs.to(device)
        attention_mask = attention_mask.to(device)

        log_likelihood = 0.0
        token_count = 0

        # BATCHED FORWARD PASSES (ONLY CHANGE)
        for i in range(0, len(masked_inputs), batch_size):

            batch_input = masked_inputs[i:i+batch_size]
            batch_mask = attention_mask[i:i+batch_size]

            with torch.no_grad():
                outputs = model(input_ids=batch_input, attention_mask=batch_mask)
                logits = outputs.logits

            for j in range(batch_input.size(0)):

                token_idx = i + j + 1
                true_token = target_tokens[i + j]

                # safety check (prevents crashes)
                if true_token >= logits.size(-1):
                    continue

                log_probs = torch.log_softmax(logits[j, token_idx], dim=-1)

                log_likelihood += log_probs[true_token].item()
                token_count += 1

        if token_count == 0:
            return None

        return math.exp(-log_likelihood / token_count)

    except:
        return None

# --------------------------------------------
# MODELS
# --------------------------------------------

models_config = {

    # BERT MODELS
    "Base_BERT": "bert-base-uncased",
    "Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models/csee_incorrect_bert",
    "LoRA_Incorrect_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora/lora_incorrect_base_bert",
    "LoRA_MUSE_BERT": "/content/drive/MyDrive/SLA_Project_CSEE/models_lora_muse/lora_incorrect_base_bert_muse",
    "MUSE_ONLY_BERT": "/content/drive/MyDrive/outputs/muse_only_uncased",
}

# --------------------------------------------
# DATASETS
# --------------------------------------------

essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# MAIN LOOP
# ============================================

for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading:", model_name)
    print("========================================")

    # ✅ MODEL TYPE HANDLING
    if "BabyRoBERTa" in model_name:
        model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
        tokenizer = roberta_tokenizer

    elif "LoRA" in model_name:
        base_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
        model = PeftModel.from_pretrained(base_model, model_path).to(device)
        tokenizer = bert_tokenizer

    else:
        model = BertForMaskedLM.from_pretrained(model_path).to(device)
        tokenizer = bert_tokenizer

    model.eval()

    results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        # Load data
        if group == "Chinese":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]

        else:
            with open(path, "r", encoding="utf-8") as f:
                essays = f.readlines()

        essays = essays[:200]

        for essay in tqdm(essays):
            essay = essay.strip()

            if len(essay.split()) < 5:
                continue

            sentences = extract_sentences_with_prepositions(essay)

            for sent in sentences:

                if len(sent.split()) < 5:
                    continue

                ppl = compute_pseudo_perplexity(sent, tokenizer, model)

                if ppl is not None:
                    ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")

        results[group] = avg_ppl
        print(f"{group}: {avg_ppl:.3f}")

    all_results[model_name] = results

# ============================================
# FINAL TABLE
# ============================================

print("\n\nFINAL COMPARISON TABLE")
print("========================================")

groups = list(essay_files.keys())

print(f"{'Model':<30}", end="")
for g in groups:
    print(f"{g:<12}", end="")
print()

for model_name in models_config.keys():
    print(f"{model_name:<30}", end="")

    for g in groups:
        val = all_results[model_name][g]

        if np.isnan(val):
            print(f"{'NaN':<12}", end="")
        else:
            print(f"{val:.2f}".ljust(12), end="")

    print()

In [ ]:
# ============================================
# FINAL — FAST + CORRECT (NO COMPROMISE)
# ============================================

import torch
import numpy as np
import re
import math
import os
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# --------------------------------------------
# DEVICE
# --------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# TOKENIZER
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# PREPOSITIONS
# --------------------------------------------
PREPOSITIONS = {
    "in","on","at","to","for","with","of","by","from",
    "about","above","across","after","against","along","among","around",
    "as","before","behind","below","beneath","beside","between","beyond",
    "during","except","inside","into","near","off","onto","out","outside",
    "over","past","since","through","throughout","toward","under","underneath",
    "until","up","upon","within","without"
}

# --------------------------------------------
# SENTENCE FILTER
# --------------------------------------------
def extract_sentences_with_prepositions(text):
    sentences = re.split(r'[.!?]', text)
    valid = []

    for sent in sentences:
        words = re.findall(r'\b\w+\b', sent.lower())

        if len(words) > 5 and any(w in PREPOSITIONS for w in words):
            valid.append(sent.strip())

    return valid

# --------------------------------------------
# 🚀 FAST PSEUDO PERPLEXITY (BATCHED)
# --------------------------------------------
def compute_pseudo_perplexity(text, model, tokenizer, batch_size=32):
    try:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=64   # SAME as your original
        )

        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]

        seq_len = input_ids.size(0)

        masked_inputs = []
        target_tokens = []

        # SAME LOGIC: mask every token
        for i in range(1, seq_len - 1):
            masked = input_ids.clone()
            target_tokens.append(input_ids[i].item())
            masked[i] = tokenizer.mask_token_id
            masked_inputs.append(masked)

        if len(masked_inputs) == 0:
            return None

        masked_inputs = torch.stack(masked_inputs)
        attention_mask = attention_mask.unsqueeze(0).repeat(len(masked_inputs), 1)

        masked_inputs = masked_inputs.to(device)
        attention_mask = attention_mask.to(device)

        log_likelihood = 0.0
        token_count = 0

        # 🚀 BATCHED FORWARD PASSES
        for i in range(0, len(masked_inputs), batch_size):
            batch_input = masked_inputs[i:i+batch_size]
            batch_mask = attention_mask[i:i+batch_size]

            with torch.no_grad():
                outputs = model(batch_input, attention_mask=batch_mask)
                logits = outputs.logits

            for j in range(batch_input.size(0)):
                token_idx = i + j + 1
                true_token = target_tokens[i + j]

                vocab_size = logits.size(-1)
                if true_token >= vocab_size:
                    continue

                log_probs = torch.log_softmax(logits[j, token_idx], dim=-1)
                log_likelihood += log_probs[true_token].item()
                token_count += 1

        if token_count == 0:
            return None

        return math.exp(-log_likelihood / token_count)

    except:
        return None

# --------------------------------------------
# DATA LOADER (UNCHANGED)
# --------------------------------------------
def load_essays(path, group):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    essays = text.split("\n")
    essays = [e.strip() for e in essays if len(e.split()) > 50]

    if len(essays) < 50:
        words = text.split()
        chunk_size = 150

        essays = [
            " ".join(words[i:i+chunk_size])
            for i in range(0, len(words), chunk_size)
        ]

    essays = essays[:200]  # ✅ KEEP ALL 200

    print(f"{group} essays count:", len(essays))
    return essays

# --------------------------------------------
# MODELS
# --------------------------------------------
models_config = {
    "BabyRoBERTa_MIX_seed42": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed42_final",
    "BabyRoBERTa_MIX_seed97": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed97_final",
    "BabyRoBERTa_MIX_seed123": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed123_final",
    "BabyRoBERTa_MIX_seed420": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed420_final",
    "BabyRoBERTa_MIX_seed999": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed999_final",

    "BabyRoBERTa_seed42": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final",
    "BabyRoBERTa_seed97": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97_final",
    "BabyRoBERTa_seed123": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final",
    "BabyRoBERTa_seed420": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420_final",
    "BabyRoBERTa_seed999": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final",
}

# --------------------------------------------
# DATASETS
# --------------------------------------------
essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# MAIN LOOP (UNCHANGED)
# ============================================
for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading:", model_name)
    print("========================================")

    torch.cuda.empty_cache()

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    results = {}

    for group, path in essay_files.items():

        if not os.path.exists(path):
            print(f"⚠️ Missing: {group}")
            continue

        print(f"\nEvaluating {group}...")

        essays = load_essays(path, group)
        ppl_scores = []

        for essay in tqdm(essays):

            sentences = extract_sentences_with_prepositions(essay)

            if len(sentences) == 0:
                sentences = [essay]

            for sent in sentences:
                ppl = compute_pseudo_perplexity(sent, model, tokenizer)

                if ppl is not None:
                    ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")
        results[group] = avg_ppl

        print(f"{group}: {avg_ppl:.3f}")

    all_results[model_name] = results

# ============================================
# FINAL TABLE
# ============================================
print("\n\nFINAL COMPARISON TABLE")
print("========================================")

groups = list(essay_files.keys())

print(f"{'Model':<30}", end="")
for g in groups:
    print(f"{g:<12}", end="")
print()

for model_name in all_results.keys():
    print(f"{model_name:<30}", end="")

    for g in groups:
        val = all_results[model_name].get(g, float("nan"))

        if np.isnan(val):
            print(f"{'NaN':<12}", end="")
        else:
            print(f"{val:.2f}".ljust(12), end="")

    print()

In [ ]:
# ============================================
# FINAL — FAST + CORRECT (NO COMPROMISE)
# ============================================

import torch
import numpy as np
import re
import math
import os
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# --------------------------------------------
# DEVICE
# --------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# TOKENIZER
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# PREPOSITIONS
# --------------------------------------------
PREPOSITIONS = {
    "in","on","at","to","for","with","of","by","from",
    "about","above","across","after","against","along","among","around",
    "as","before","behind","below","beneath","beside","between","beyond",
    "during","except","inside","into","near","off","onto","out","outside",
    "over","past","since","through","throughout","toward","under","underneath",
    "until","up","upon","within","without"
}

# --------------------------------------------
# SENTENCE FILTER
# --------------------------------------------
def extract_sentences_with_prepositions(text):
    sentences = re.split(r'[.!?]', text)
    valid = []

    for sent in sentences:
        words = re.findall(r'\b\w+\b', sent.lower())

        if len(words) > 5 and any(w in PREPOSITIONS for w in words):
            valid.append(sent.strip())

    return valid

# --------------------------------------------
#  FAST PSEUDO PERPLEXITY (BATCHED)
# --------------------------------------------
def compute_pseudo_perplexity(text, model, tokenizer, batch_size=32):
    try:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=64   # SAME as your original
        )

        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]

        seq_len = input_ids.size(0)

        masked_inputs = []
        target_tokens = []

        # SAME LOGIC: mask every token
        for i in range(1, seq_len - 1):
            masked = input_ids.clone()
            target_tokens.append(input_ids[i].item())
            masked[i] = tokenizer.mask_token_id
            masked_inputs.append(masked)

        if len(masked_inputs) == 0:
            return None

        masked_inputs = torch.stack(masked_inputs)
        attention_mask = attention_mask.unsqueeze(0).repeat(len(masked_inputs), 1)

        masked_inputs = masked_inputs.to(device)
        attention_mask = attention_mask.to(device)

        log_likelihood = 0.0
        token_count = 0

        #  BATCHED FORWARD PASSES
        for i in range(0, len(masked_inputs), batch_size):
            batch_input = masked_inputs[i:i+batch_size]
            batch_mask = attention_mask[i:i+batch_size]

            with torch.no_grad():
                outputs = model(batch_input, attention_mask=batch_mask)
                logits = outputs.logits

            for j in range(batch_input.size(0)):
                token_idx = i + j + 1
                true_token = target_tokens[i + j]

                vocab_size = logits.size(-1)
                if true_token >= vocab_size:
                    continue

                log_probs = torch.log_softmax(logits[j, token_idx], dim=-1)
                log_likelihood += log_probs[true_token].item()
                token_count += 1

        if token_count == 0:
            return None

        return math.exp(-log_likelihood / token_count)

    except:
        return None

# --------------------------------------------
# DATA LOADER (UNCHANGED)
# --------------------------------------------
def load_essays(path, group):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    essays = text.split("\n")
    essays = [e.strip() for e in essays if len(e.split()) > 50]

    if len(essays) < 50:
        words = text.split()
        chunk_size = 150

        essays = [
            " ".join(words[i:i+chunk_size])
            for i in range(0, len(words), chunk_size)
        ]

    essays = essays[:200]  # ✅ KEEP ALL 200

    print(f"{group} essays count:", len(essays))
    return essays

# --------------------------------------------
# MODELS
# --------------------------------------------
models_config = {
    "30M_plus5M_seed42": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed42_final",
    "30M_plus5M_seed97": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed97_final",
    "30M_plus5M_seed123": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed123_final",
    "30M_plus5M_seed420": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed420_final",
    "30M_plus5M_seed999": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_plus5M_seed999_final",
}

# --------------------------------------------
# DATASETS
# --------------------------------------------
essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# MAIN LOOP (UNCHANGED)
# ============================================
for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading:", model_name)
    print("========================================")

    torch.cuda.empty_cache()

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    results = {}

    for group, path in essay_files.items():

        if not os.path.exists(path):
            print(f"⚠️ Missing: {group}")
            continue

        print(f"\nEvaluating {group}...")

        essays = load_essays(path, group)
        ppl_scores = []

        for essay in tqdm(essays):

            sentences = extract_sentences_with_prepositions(essay)

            if len(sentences) == 0:
                sentences = [essay]

            for sent in sentences:
                ppl = compute_pseudo_perplexity(sent, model, tokenizer)

                if ppl is not None:
                    ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")
        results[group] = avg_ppl

        print(f"{group}: {avg_ppl:.3f}")

    all_results[model_name] = results

# ============================================
# FINAL TABLE
# ============================================
print("\n\nFINAL COMPARISON TABLE")
print("========================================")

groups = list(essay_files.keys())

print(f"{'Model':<30}", end="")
for g in groups:
    print(f"{g:<12}", end="")
print()

for model_name in all_results.keys():
    print(f"{model_name:<30}", end="")

    for g in groups:
        val = all_results[model_name].get(g, float("nan"))

        if np.isnan(val):
            print(f"{'NaN':<12}", end="")
        else:
            print(f"{val:.2f}".ljust(12), end="")

    print()

In [ ]:
# ============================================
# FINAL — PREPOSITION-AWARE + FAST + CORRECT
# ============================================

import torch
import numpy as np
import re
import math
import os
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# --------------------------------------------
# DEVICE
# --------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# TOKENIZER
# --------------------------------------------
TOKENIZER_DIR = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_DIR)

# --------------------------------------------
# PREPOSITIONS
# --------------------------------------------
PREPOSITIONS = {
    "in","on","at","to","for","with","of","by","from",
    "about","above","across","after","against","along","among","around",
    "as","before","behind","below","beneath","beside","between","beyond",
    "during","except","inside","into","near","off","onto","out","outside",
    "over","past","since","through","throughout","toward","under","underneath",
    "until","up","upon","within","without"
}

# --------------------------------------------
# SENTENCE FILTER (FAIR VERSION)
# --------------------------------------------
def extract_sentences_with_prepositions(text):
    sentences = re.split(r'[.!?]', text)

    preposition_sentences = []
    fallback_sentences = []

    for sent in sentences:
        words = re.findall(r'\b\w+\b', sent.lower())

        if len(words) > 5:
            fallback_sentences.append(sent.strip())

            if any(w in PREPOSITIONS for w in words):
                preposition_sentences.append(sent.strip())

    # ✅ fairness fix
    if len(preposition_sentences) > 0:
        return preposition_sentences
    else:
        return fallback_sentences


# --------------------------------------------
# 🚀 FIXED PSEUDO PERPLEXITY
# --------------------------------------------
def compute_pseudo_perplexity(text, model, tokenizer, max_length=128, batch_size=32):

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = enc["input_ids"][0]
    attention_mask = enc["attention_mask"][0]

    seq_len = input_ids.size(0)

    if seq_len < 3:
        return None

    masked_inputs = []
    target_tokens = []

    # mask each token (except special tokens)
    for i in range(1, seq_len - 1):
        masked = input_ids.clone()
        masked[i] = tokenizer.mask_token_id

        masked_inputs.append(masked)
        target_tokens.append(input_ids[i].item())

    masked_inputs = torch.stack(masked_inputs)
    attention_mask = attention_mask.unsqueeze(0).repeat(len(masked_inputs), 1)

    masked_inputs = masked_inputs.to(device)
    attention_mask = attention_mask.to(device)

    log_likelihood = 0.0
    token_count = 0

    # batching
    for batch_start in range(0, len(masked_inputs), batch_size):

        batch_input = masked_inputs[batch_start:batch_start+batch_size]
        batch_mask = attention_mask[batch_start:batch_start+batch_size]

        with torch.no_grad():
            outputs = model(batch_input, attention_mask=batch_mask)
            logits = outputs.logits

        for j in range(batch_input.size(0)):

            original_idx = batch_start + j
            token_position = original_idx + 1

            true_token = target_tokens[original_idx]

            if true_token >= logits.size(-1):
                continue

            log_probs = torch.log_softmax(logits[j, token_position], dim=-1)

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)


# --------------------------------------------
# DATA LOADER
# --------------------------------------------
def load_essays(path, group):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    essays = text.split("\n")
    essays = [e.strip() for e in essays if len(e.split()) > 20]

    if len(essays) < 50:
        words = text.split()
        chunk_size = 150

        essays = [
            " ".join(words[i:i+chunk_size])
            for i in range(0, len(words), chunk_size)
        ]

    essays = essays[:200]

    print(f"{group} essays:", len(essays))
    return essays


# --------------------------------------------
# MODELS
# --------------------------------------------
models_config = {
    "BabyRoBERTa_MIX_seed42": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed42_final",
    "BabyRoBERTa_MIX_seed97": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed97_final",
    "BabyRoBERTa_MIX_seed123": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed123_final",
    "BabyRoBERTa_MIX_seed420": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed420_final",
    "BabyRoBERTa_MIX_seed999": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed999_final",

    "BabyRoBERTa_seed42": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final",
    "BabyRoBERTa_seed97": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97_final",
    "BabyRoBERTa_seed123": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final",
    "BabyRoBERTa_seed420": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420_final",
    "BabyRoBERTa_seed999": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final",


}

# --------------------------------------------
# DATASETS
# --------------------------------------------
essay_files = {
    "Native": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

all_results = {}

# ============================================
# MAIN LOOP
# ============================================
for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Loading:", model_name)
    print("========================================")

    torch.cuda.empty_cache()

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    results = {}

    for group, path in essay_files.items():

        if not os.path.exists(path):
            print(f"⚠️ Missing: {group}")
            continue

        print(f"\nEvaluating {group}...")

        essays = load_essays(path, group)
        ppl_scores = []

        for essay in tqdm(essays):

            sentences = extract_sentences_with_prepositions(essay)

            if len(sentences) == 0:
                sentences = [essay]

            for sent in sentences:
                ppl = compute_pseudo_perplexity(sent, model, tokenizer)

                if ppl is not None:
                    ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores) if len(ppl_scores) > 0 else float("nan")
        results[group] = avg_ppl

        print(f"{group}: {avg_ppl:.3f}")

    all_results[model_name] = results


# ============================================
# FINAL TABLE
# ============================================
print("\n\nFINAL COMPARISON TABLE")
print("========================================")

groups = list(essay_files.keys())

print(f"{'Model':<30}", end="")
for g in groups:
    print(f"{g:<12}", end="")
print()

for model_name in all_results.keys():
    print(f"{model_name:<30}", end="")

    for g in groups:
        val = all_results[model_name].get(g, float("nan"))

        if np.isnan(val):
            print(f"{'NaN':<12}", end="")
        else:
            print(f"{val:.2f}".ljust(12), end="")

    print()

In [ ]:
import numpy as np

# -----------------------------
# BabyRoBERTa MIX
# -----------------------------
mix_data = {
    "Native":   [119.179, 84.287, 104.424, 148.308, 137.311],
    "Chinese":  [162.258, 119.110, 143.980, 227.231, 188.578],
    "Japanese": [95.252, 67.160, 83.361, 113.654, 111.061],
    "Korean":   [183.926, 139.308, 163.186, 264.020, 209.437],
    "Thai":     [85.916, 59.283, 73.908, 93.957, 98.458],
    "Filipino": [89.584, 63.164, 76.105, 100.804, 104.133],
    "Urdu":     [142.586, 104.968, 126.780, 191.309, 167.945],
}

# -----------------------------
# BabyRoBERTa W2W
# -----------------------------
w2w_data = {
    "Native":   [57.001, 57.045, 54.711, 56.501, 54.235],
    "Chinese":  [83.526, 91.622, 84.084, 84.839, 86.854],
    "Japanese": [50.884, 51.035, 49.948, 52.204, 48.742],
    "Korean":   [90.392, 102.712, 93.642, 93.540, 102.921],
    "Thai":     [41.721, 40.144, 39.931, 42.347, 37.718],
    "Filipino": [41.571, 39.761, 39.205, 42.092, 38.721],
    "Urdu":     [62.279, 64.459, 62.556, 64.002, 65.488],
}

def compute_avg(data, exclude=None):
    if exclude is None:
        exclude = []
    result = {}
    for lang, values in data.items():
        if lang not in exclude:
            result[lang] = round(np.mean(values), 2)
    return result

# exclude Korean and Urdu if needed
exclude_langs = ["Korean", "Urdu"]

mix_avg = compute_avg(mix_data, exclude=exclude_langs)
w2w_avg = compute_avg(w2w_data, exclude=exclude_langs)

print("BabyRoBERTa MIX average:")
for k, v in mix_avg.items():
    print(f"{k}: {v}")

print("\nBabyRoBERTa W2W average:")
for k, v in w2w_avg.items():
    print(f"{k}: {v}")

print("\nLaTeX rows:")
print(f"BabyRoBERTa (30M-W2W) & {w2w_avg['Native']} & {w2w_avg['Chinese']} & {w2w_avg['Japanese']} & {w2w_avg['Thai']} & {w2w_avg['Filipino']} \\\\")
print(f"BabyRoBERTa (30M-Mix) & {mix_avg['Native']} & {mix_avg['Chinese']} & {mix_avg['Japanese']} & {mix_avg['Thai']} & {mix_avg['Filipino']} \\\\")

In [ ]:
# ============================================
# PERPLEXITY CHECK (ALL MODELS)
# ============================================

import torch
import numpy as np
import re
import math
import os
from tqdm import tqdm
from transformers import RobertaForMaskedLM, RobertaTokenizerFast

# --------------------------------------------
# DEVICE
# --------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# TOKENIZER (SAME FOR ALL BABYROBERTA)
# --------------------------------------------
TOKENIZER_PATH = "/content/drive/MyDrive/MUSE_artifacts/babyroberta_tokenizer"
tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_PATH)

# --------------------------------------------
# MODELS (ALL YOU WANT TO TEST)
# --------------------------------------------
models_config = {
    "BabyRoBERTa_seed42": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed42_final",
    "BabyRoBERTa_seed97": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed97_final",
    "BabyRoBERTa_seed123": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed123_final",
    "BabyRoBERTa_seed420": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed420_final",
    "BabyRoBERTa_seed999": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_seed999_final",

    "BabyRoBERTa_MIX_seed42": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed42_final",
    "BabyRoBERTa_MIX_seed97": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed97_final",
    "BabyRoBERTa_MIX_seed123": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed123_final",
    "BabyRoBERTa_MIX_seed420": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed420_final",
    "BabyRoBERTa_MIX_seed999": "/content/drive/MyDrive/MUSE_artifacts/babyroberta_30M_mixed_seed999_final",
}

# --------------------------------------------
# PSEUDO PERPLEXITY
# --------------------------------------------
def compute_pseudo_perplexity(text, model, max_length=128, batch_size=32):

    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)

    input_ids = enc["input_ids"][0]
    attention_mask = enc["attention_mask"][0]

    seq_len = input_ids.size(0)
    if seq_len < 3:
        return None

    masked_inputs = []
    target_tokens = []

    for i in range(1, seq_len - 1):
        masked = input_ids.clone()
        masked[i] = tokenizer.mask_token_id
        masked_inputs.append(masked)
        target_tokens.append(input_ids[i].item())

    masked_inputs = torch.stack(masked_inputs)
    attention_mask = attention_mask.unsqueeze(0).repeat(len(masked_inputs), 1)

    masked_inputs = masked_inputs.to(device)
    attention_mask = attention_mask.to(device)

    log_likelihood = 0.0
    token_count = 0

    for batch_start in range(0, len(masked_inputs), batch_size):

        batch_input = masked_inputs[batch_start:batch_start+batch_size]
        batch_mask = attention_mask[batch_start:batch_start+batch_size]

        with torch.no_grad():
            outputs = model(batch_input, attention_mask=batch_mask)
            logits = outputs.logits

        for j in range(batch_input.size(0)):
            original_idx = batch_start + j
            token_position = original_idx + 1
            true_token = target_tokens[original_idx]

            log_probs = torch.log_softmax(logits[j, token_position], dim=-1)

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)


# --------------------------------------------
# LOAD JAPANESE DATA
# --------------------------------------------
file_path = "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

words = text.split()
chunk_size = 150

essays = [
    " ".join(words[i:i+chunk_size])
    for i in range(0, len(words), chunk_size)
]

essays = essays[:200]
print("Total essays:", len(essays))


# ============================================
# MAIN LOOP OVER MODELS
# ============================================
results = {}

for model_name, model_path in models_config.items():

    print("\n========================================")
    print("Model:", model_name)
    print("========================================")

    torch.cuda.empty_cache()

    model = RobertaForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    # --------------------------------------------
    # BEFORE FIX
    # --------------------------------------------
    ppl_before = []

    for essay in tqdm(essays, desc="Before"):

        ppl = compute_pseudo_perplexity(essay, model)

        if ppl is not None:
            ppl_before.append(ppl)

    avg_before = np.mean(ppl_before)

    # --------------------------------------------
    # AFTER FIX
    # --------------------------------------------
    ppl_after = []

    for essay in tqdm(essays, desc="After"):

        sentences = re.split(r'[.!?]', essay)
        sentences = [s.strip() for s in sentences if len(s.split()) > 5]

        if len(sentences) == 0:
            sentences = [essay]

        for sent in sentences:
            ppl = compute_pseudo_perplexity(sent, model)

            if ppl is not None:
                ppl_after.append(ppl)

    avg_after = np.mean(ppl_after)

    results[model_name] = (avg_before, avg_after)

    print(f"\nBefore: {avg_before:.3f}")
    print(f"After : {avg_after:.3f}")
    print(f"Diff  : {abs(avg_before - avg_after):.3f}")


# ============================================
# FINAL TABLE
# ============================================
print("\n\n===== FINAL COMPARISON =====")

for model_name, (b, a) in results.items():
    print(f"{model_name:<30} Before: {b:.2f} | After: {a:.2f} | Diff: {abs(b-a):.2f}")